# Chuyên đề 04 · Học tăng cường từ phản hồi của người (RLHF): mô hình phần thưởng, PPO và DPO trên thơ lục bát

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd04-rlhf.ipynb)

Notebook đi kèm [Chuyên đề 04 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rlhf/). Bài 26 đã hậu huấn luyện mô hình đọc số bằng một
bộ chấm viết tay. Với chatbot thật thì không có bộ chấm nào như vậy: người ta chỉ có những lần người chấm **so sánh**
hai câu trả lời. Chuyên đề này đi hết quy trình RLHF mà InstructGPT dùng, ở kích thước chạy được trên CPU:
một GPT nhỏ viết câu tám chữ nối tiếp một câu sáu chữ của *Truyện Kiều*, người chấm mô phỏng chọn câu hay hơn trong
từng cặp, một mô hình phần thưởng học từ các lần chọn đó, rồi PPO tối ưu mô hình theo phần thưởng ấy. DPO thì bỏ qua
mô hình phần thưởng. "Gu" của người chấm ở đây là một chương trình (luật lục bát), nên ta đo được điều mà RLHF thật
không đo được: chất lượng *thật* của từng câu thơ, và lúc nào việc tối ưu bắt đầu đi sai hướng.

**Bạn sẽ làm:**
1. Khớp mô hình Bradley–Terry: từ các lần so sánh ra điểm số, và điều điểm số không thể cho biết.
2. Huấn luyện một mô hình phần thưởng từ 1.000 lần so sánh, rồi so với trần độ chính xác do người chấm ồn đặt ra.
3. Thấy định luật Goodhart: trên một bảng tính chính xác, rồi với best-of-N và PPO trên thơ lục bát, và đọc những câu
   mà mô hình viết ra để chiều lòng mô hình phần thưởng.
4. Tự lắp PPO từng mảnh (phạt KL theo từng token, GAE, tỉ số bị kẹp) và thấy vì sao đặt KL vào hàm mất mát sai cách
   thì nó không còn kéo mô hình lại.
5. Chạy DPO và IPO: chúng đi về đâu khi người chấm không bao giờ đổi ý, và vì sao xác suất của câu *được chọn* cũng có
   thể giảm.

**Cần biết trước:** [Bài 26 · Từ mô hình ngôn ngữ đến trợ lý](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/)
(π*, DPO trên một bảng, best-of-N), [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/)
và [Chuyên đề 03](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/nghien-cuu-sinh-va-hau-huan-luyen/) (SFT và học tăng
cường quên khác nhau thế nào).
**Thời gian chạy:** khoảng 1 phút trên CPU, kể cả tải dữ liệu lần đầu.

Notebook dùng **PyTorch**. Mỗi mục chạy trực tiếp một phiên bản ngắn, rồi nạp kết quả đầy đủ (3 seed, 200 vòng PPO,
6 lượt DPO) mà script `reel/rlhf/sweep.py` của khóa học đã tính sẵn, cùng mã và cùng seed.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

In [ ]:
import base64
import copy
import hashlib
import json
import math
import os
import re
import unicodedata
import urllib.request
from itertools import pairwise
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=1):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def vn_tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Nhãn trục kiểu Việt Nam; gọi ngay trước plt.show(). Trục log ngắn lấy vạch 1–2–5, trục log dài lấy lũy thừa của 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))

## 2 · Dữ liệu

*Truyện Kiều* từ Wikisource, ghim ở phiên bản 205132 như Bài 15 và 17 (có kiểm MD5), và hai tệp của script:
`rlhf-lucbat.json` chứa kết quả của mọi lần chạy, `rlhf-weights.json` chứa trọng số float16 của ba mô hình (mô hình
SFT, mô hình phần thưởng và một chính sách PPO) cùng 2.500 cặp câu đã được người chấm mô phỏng so sánh.

In [ ]:
KIEU_SRC = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"
KIEU_RAW = Path("truyen_kieu_205132.wiki")
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "HocSauReels-notebook/1.0 (education; https://d3qr1cjln6qqtk.cloudfront.net/)")]
urllib.request.install_opener(opener)  # Wikimedia từ chối các yêu cầu không có User-Agent
if not KIEU_RAW.exists():
    urllib.request.urlretrieve(KIEU_SRC, KIEU_RAW)
assert hashlib.md5(KIEU_RAW.read_bytes()).hexdigest() == "d4090daac094841888ee2b2b18661065"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


RUNS = fetch_json("rlhf-lucbat.json", "2ba3eaad71495e13bd29c03d0fe4907d")
W = fetch_json("rlhf-weights.json", "5be6aa64330bcdba4008ddeebbe0450d")

Mỗi cặp câu 6–8 của truyện là một ví dụ: **lời nhắc** là câu sáu chữ, **câu trả lời** là câu tám chữ ngay sau nó.
Đơn vị (token) là âm tiết, viết thường, bỏ dấu câu. Ta chia theo vị trí trong truyện: 1.300 cặp đầu để huấn luyện,
163 cặp tiếp theo để kiểm định, 164 cặp cuối để kiểm tra.

In [ ]:
body = KIEU_RAW.read_bytes().decode("utf-8").split("<poem>")[1].split("</poem>")[0]
raw = [re.sub(r"\{\{số\|\d+\}\}", "", ln).strip() for ln in body.split("\n")]
text = "\n".join(ln for ln in raw if ln) + "\n"
assert hashlib.md5(text.encode()).hexdigest() == "dd94ee390bfc543287b11af62b2f2fc8"
lines = [re.sub(r"[^\w\s]", " ", ln).lower().split() for ln in text[:-1].split("\n")]
couplets = [(a, b) for a, b in pairwise(lines) if len(a) == 6 and len(b) == 8]
PAD, SEP, EOS = 0, 1, 2
VOCAB = ["<pad>", "<sep>", "<eos>", *sorted({s for ln in lines for s in ln})]
IX = {s: i for i, s in enumerate(VOCAB)}
V = len(VOCAB)
X = torch.tensor([[IX[s] for s in a] + [SEP] + [IX[s] for s in b] + [EOS] for a, b in couplets])
X_train, X_val, X_test = X[:1300], X[1300:1463], X[1463:]
PLEN, RESP = 7, 13  # lời nhắc: 6 âm tiết + <sep>; câu trả lời: tối đa 13 token (âm tiết, rồi <eos> nếu có)
T_MAX = PLEN + RESP
print(f"{vn(len(lines), 0)} câu thơ, {vn(len(couplets), 0)} cặp 6–8, {vn(V, 0)} token (kể cả <pad>, <sep>, <eos>)")
print(f"huấn luyện {vn(len(X_train), 0)} · kiểm định {len(X_val)} · kiểm tra {len(X_test)} cặp")
for a, b in couplets[:2]:
    print("  ", " ".join(a), "/", " ".join(b))
assert (len(lines), len(couplets), V) == (3254, 1627, 2385)

## 3 · RLHF từ 2017 đến nay

Quy trình mà chuyên đề này dựng lại có ba bước (Ouyang và cộng sự, 2022, gọi là InstructGPT):

1. **SFT.** Tinh chỉnh mô hình đã tiền huấn luyện trên các câu trả lời mẫu. Kết quả là $\pi_{\text{ref}}$.
2. **Mô hình phần thưởng.** Cho $\pi_{\text{ref}}$ viết nhiều câu trả lời, người chấm xếp hạng chúng, rồi huấn luyện
   một mô hình $r_\phi(x, y)$ dự đoán câu nào được chọn.
3. **Học tăng cường.** Tối ưu $\pi$ để có $r_\phi$ cao mà không đi quá xa $\pi_{\text{ref}}$:
   $\max_\pi \; \mathbb{E}_{y\sim\pi}[r_\phi(x, y)] - \beta\,\mathrm{KL}(\pi\,\|\,\pi_{\text{ref}})$, bằng PPO.

Vài mốc chính:
- **Christiano và cộng sự (2017)** dạy tác tử chơi Atari và điều khiển robot mô phỏng chỉ từ những lần người xem so
  sánh hai đoạn video ngắn. Đây là nơi mô hình phần thưởng học từ so sánh ra đời.
- **Ziegler và cộng sự (2019), Stiennon và cộng sự (2020)** đưa cách đó vào mô hình ngôn ngữ (viết tiếp, tóm tắt), với
  phạt KL theo từng token.
- **InstructGPT (2022)**: khoảng 13 nghìn lời nhắc cho SFT, 33 nghìn cho mô hình phần thưởng, 31 nghìn cho PPO; người
  chấm xếp hạng 4 đến 9 câu trả lời một lúc; mô hình phần thưởng 6 tỉ tham số. Người chấm thích câu trả lời của mô hình
  1,3 tỉ tham số sau RLHF hơn của GPT-3 175 tỉ. Hai người chấm chỉ đồng ý với nhau 72,6–77,3% số lần, và mô hình
  phần thưởng đoán đúng lựa chọn của họ 69,6–72,4%: dữ liệu so sánh vốn rất ồn.
- **Bai và cộng sự (2022)** huấn luyện trợ lý "hữu ích và vô hại" theo cùng quy trình; *Constitutional AI* thay một
  phần người chấm bằng chính mô hình chấm theo một bộ nguyên tắc (RLAIF).
- **DPO (Rafailov và cộng sự, 2023)** bỏ hẳn mô hình phần thưởng và học thẳng từ các cặp (mục 10).
- **Llama 3 (2024)** lặp nhiều vòng: mô hình phần thưởng → lấy mẫu từ chối (best-of-N) → SFT → DPO. **Tülu 3 (2024)**:
  SFT → DPO → học tăng cường với phần thưởng kiểm chứng được (RLVR).
- **DeepSeek-R1 (2025)** dùng GRPO với phần thưởng là các luật kiểm tra đáp án; trong lúc học tăng cường, điểm pass@1
  của DeepSeek-R1-Zero trên AIME 2024 tăng từ 15,6% lên 71,0%.

Bài 26 đã làm phần "học tăng cường với bộ chấm viết tay" (kiểu GRPO), lời giải đóng $\pi^*$, DPO trên một bảng, và
best-of-N. Chuyên đề này thêm phần Bài 26 không có: **phần thưởng học được từ so sánh ồn**, PPO đầy đủ, và cái giá
của việc tối ưu theo một phần thưởng chỉ gần đúng.

## 4 · Từ so sánh đến điểm số: mô hình Bradley–Terry

Người chấm không cho điểm, họ chỉ chọn. Mô hình Bradley–Terry nối hai thứ đó: mỗi câu trả lời có một điểm $r$, và
$$P(y_a \succ y_b) = \sigma\big(r(y_a) - r(y_b)\big), \qquad
  \mathcal{L}(r) = -\sum_{\text{cặp}} \log \sigma\big(r(y_{\text{thắng}}) - r(y_{\text{thua}})\big).$$
Chỉ **hiệu** hai điểm xuất hiện trong công thức. Ô dưới khớp mô hình cho 6 câu trả lời của hai lời nhắc (A: câu 0–2,
B: câu 3–5), với 4.000 lần so sánh mô phỏng, luôn giữa hai câu của *cùng* một lời nhắc như trong RLHF thật. Ta khớp hai
lần từ hai điểm xuất phát khác nhau và so với "độ hay thật" $u$ mà người chấm dùng.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def fit_bt(i, j, y, r0, steps=3000, lr=2.0):
    """Bradley–Terry bằng gradient descent; y = 1 khi câu i thắng câu j."""
    r = r0.astype(float).copy()
    for _ in range(steps):
        e = sigmoid(r[i] - r[j]) - y  # đạo hàm của mất mát theo (r_i − r_j), từng cặp
        r -= lr * (np.bincount(i, e, len(r)) - np.bincount(j, e, len(r))) / len(y)
    return r


def bt_nll(r, i, j, y):
    d = r[i] - r[j]
    return float(np.mean(np.logaddexp(0, -d) * y + np.logaddexp(0, d) * (1 - y)))


u = np.array([0.2, 1.0, 1.6, 0.4, 0.7, 2.0])
prompt_of = np.array([0, 0, 0, 1, 1, 1])
rng = np.random.default_rng(0)
grp = rng.integers(0, 2, 4000)
i = 3 * grp + rng.integers(0, 3, 4000)
j = 3 * grp + (i - 3 * grp + rng.integers(1, 3, 4000)) % 3  # một câu khác của cùng lời nhắc
y = (rng.random(4000) < sigmoid(u[i] - u[j])).astype(float)
fit_a = fit_bt(i, j, y, np.zeros(6))
fit_b = fit_bt(i, j, y, np.array([5.0, 5, 5, -5, -5, -5]))
center = lambda r: r - np.array([r[prompt_of == p].mean() for p in prompt_of])
print("câu  lời nhắc   u    khớp từ 0   khớp từ ±5   khớp − trung bình lời nhắc   u − trung bình")
for k in range(6):
    print(
        f"{k:3}      {'AB'[prompt_of[k]]}    {vn(u[k], 1):>4}   {vn(fit_a[k]):>8}   {vn(fit_b[k]):>9}"
        f"   {vn(center(fit_a)[k]):>18}   {vn(center(u)[k]):>14}"
    )
print("mất mát:", vn(bt_nll(fit_a, i, j, y), 4), "và", vn(bt_nll(fit_b, i, j, y), 4))
assert abs(bt_nll(fit_a, i, j, y) - bt_nll(fit_b, i, j, y)) < 1e-9
assert np.allclose(center(fit_a), center(fit_b)) and np.abs(center(fit_a) - center(u)).max() < 0.15

Hai lần khớp có cùng mất mát đến 4 chữ số, nhưng điểm của chúng khác nhau đúng 10 giữa hai lời nhắc. Sau khi trừ trung
bình của từng lời nhắc, cả hai trùng nhau và gần với $u$. Gradient không bao giờ đổi được trung bình đó: tổng gradient
trên các câu của một lời nhắc luôn bằng 0, vì mỗi cặp cộng $e$ vào câu này và trừ $e$ khỏi câu kia.

Hệ quả cho RLHF:
- **Điểm chỉ so được giữa các câu trả lời của cùng một lời nhắc.** Mô hình phần thưởng mạng nơ-ron cũng vậy: phần
  chênh giữa các lời nhắc là tùy ý. Cộng một hằng số theo lời nhắc vào phần thưởng không đổi chính sách tối ưu, nhưng
  làm gradient ồn hơn. Vì thế PPO cần mô hình giá trị, còn RLOO và GRPO trừ trung bình của nhóm câu trả lời cùng lời
  nhắc.
- **Thang đo đến từ độ ồn của người chấm.** Mô hình học được $s\cdot u$, trong đó $s$ cho biết người chấm phân biệt rõ
  đến đâu. Người chấm càng ồn, các điểm càng sát nhau. Vì thế ta chuẩn hóa mọi phần thưởng (mục 8) để cùng một β có cùng
  ý nghĩa với mọi mô hình phần thưởng.

## 5 · Bài toán: viết câu bát

### Luật lục bát làm "gu" của người chấm

Trong RLHF thật, điều người ta muốn không viết ra được thành chương trình. Ở đây ta *giả vờ* rằng người chấm chỉ quan
tâm đến luật lục bát, viết thành 7 phép kiểm tra cho câu trả lời $y$ (câu bát) sau lời nhắc $x$ (câu lục):
1. $y$ có đúng 8 âm tiết;
2. đến 5. chữ thứ 2, 4, 6, 8 lần lượt là **bằng, trắc, bằng, bằng** (thanh bằng: không dấu hoặc dấu huyền; thanh
   trắc: sắc, hỏi, ngã, nặng);
6. chữ thứ 6 của $y$ **vần** với chữ thứ 6 của $x$ (vần thông: cùng âm cuối, và cùng nhóm nguyên âm, trừ khi âm cuối
   là *nh*, *ch* hay *i* thì nguyên âm nào cũng được);
7. chữ thứ 6 và thứ 8 của $y$ có một chữ thanh huyền và một chữ thanh ngang.

Điểm **gold** $g \in [0, 1]$ là tỉ lệ phép kiểm tra đạt. Đó là "chất lượng thật" mà ta dùng để *đo*, không bao giờ để
huấn luyện. Ô dưới viết phần tách thanh và vần, rồi thử trên vài chữ.

In [ ]:
TONE = {"̀": "huyền", "́": "sắc", "̃": "ngã", "̉": "hỏi", "̣": "nặng"}  # dấu, dạng NFD
ONSET = re.compile(r"^(ngh|ng|nh|ch|gh|gi(?=[aeiouyăâêôơư])|kh|ph|th|tr|qu|[bcdđghklmnpqrstvx])")
CODAS = ("ng", "nh", "ch", "c", "m", "n", "p", "t", "i", "y", "o", "u")
NUCLEUS = {"y": "i", "yê": "iê", "ia": "iê", "ya": "iê", "ua": "uô", "ưa": "ươ"}  # các cách viết của cùng một nguyên âm
GROUP = {"i": "F", "ê": "F", "e": "F", "iê": "F", "u": "B", "ô": "B", "o": "B", "uô": "B",
         "a": "A", "ă": "A", "â": "A", "ơ": "A", "ư": "A", "ươ": "A"}  # fmt: skip
FREE_CODA = {"nh", "ch", "i"}  # inh ~ anh ~ ênh, ai ~ ơi ~ ươi: nguyên âm nào cũng vần


def tone(s):
    marks = [TONE[c] for c in unicodedata.normalize("NFD", s) if c in TONE]
    return marks[0] if marks else "ngang"


def rime(s):
    """(nguyên âm, âm cuối) của một âm tiết, đã bỏ dấu thanh: phần tạo nên vần."""
    bare = unicodedata.normalize("NFC", "".join(c for c in unicodedata.normalize("NFD", s) if c not in TONE))
    r = ONSET.sub("", bare)
    if len(r) > 1 and r[0] in "ou" and r[1] in "aăâeêyơ":  # âm đệm o/u: hoa, thuyền, quân
        r = r[1:]
    nuc, coda = r, ""
    for c in CODAS:
        if r.endswith(c) and len(r) > len(c):
            nuc, coda = r[: -len(c)], c
            break
    return NUCLEUS.get(nuc, nuc), {"y": "i", "o": "u"}.get(coda, coda)


def rhymes(a, b):
    (na, ca), (nb, cb) = rime(a), rime(b)
    return ca == cb and (ca in FREE_CODA or GROUP.get(na, na) == GROUP.get(nb, nb))


for s in ("đường", "thuyền", "lặng", "hoa"):
    print(f"{s:7} thanh {tone(s):6} vần {rime(s)}")
for a, b in (("thương", "đường"), ("tình", "xanh"), ("hoa", "người")):
    print(f"{a} – {b}: {'vần' if rhymes(a, b) else 'không vần'}")
assert rhymes("thương", "đường") and rhymes("tình", "xanh") and not rhymes("hoa", "người")

Để chấm hàng nghìn câu một lúc, ta tính trước cho mỗi token: thanh bằng hay trắc, có phải thanh huyền, âm cuối và nhóm
nguyên âm. Bảy phép kiểm tra khi đó chỉ là phép tra bảng trên tensor. Ô dưới chấm chính *Truyện Kiều*: nếu luật viết
đúng, gần như mọi câu bát thật phải đạt cả bảy.

In [ ]:
SYL = torch.tensor([k > EOS for k in range(V)])
tones = [tone(s) if k > EOS else "" for k, s in enumerate(VOCAB)]
BANG = torch.tensor([t in ("ngang", "huyền") for t in tones]) & SYL
HUYEN = torch.tensor([t == "huyền" for t in tones])
rimes = [rime(s) if k > EOS else ("", "") for k, s in enumerate(VOCAB)]
codas, groups = sorted({c for _, c in rimes}), sorted({GROUP.get(n, n) for n, _ in rimes})
CODA = torch.tensor([codas.index(c) for _, c in rimes])
VGROUP = torch.tensor([groups.index(GROUP.get(n, n)) for n, _ in rimes])
FREE = torch.tensor([c in FREE_CODA for _, c in rimes])
CHECKS = ["8 âm tiết", "chữ 2 bằng", "chữ 4 trắc", "chữ 6 bằng", "chữ 8 bằng", "vần chữ 6", "6–8 huyền/ngang"]


def resp_len(seq):
    """Số âm tiết trước <eos>/<pad> đầu tiên của mỗi câu trả lời."""
    r = seq[:, PLEN:]
    return ((r != EOS) & (r != PAD)).int().cumprod(1).sum(1)


def checks(seq):
    """(B, 7) giá trị đúng/sai: 7 phép kiểm tra của từng câu trả lời (seq = lời nhắc, <sep>, câu trả lời, phần đệm)."""
    r = torch.cat([seq[:, PLEN:], torch.full((len(seq), max(0, 8 - seq.shape[1] + PLEN)), PAD)], 1)
    n = resp_len(seq)
    y2, y4, y6, y8, x6 = r[:, 1], r[:, 3], r[:, 5], r[:, 7], seq[:, 5]
    return torch.stack([
        n == 8,
        (n >= 2) & BANG[y2],
        (n >= 4) & SYL[y4] & ~BANG[y4],
        (n >= 6) & BANG[y6],
        (n >= 8) & BANG[y8],
        (n >= 6) & (CODA[x6] == CODA[y6]) & (FREE[y6] | (VGROUP[x6] == VGROUP[y6])),
        (n >= 8) & BANG[y6] & BANG[y8] & (HUYEN[y6] != HUYEN[y8]),
    ], 1)  # fmt: skip


def gold(seq):
    return checks(seq).float().mean(1)


def padded(x):
    return torch.cat([x, torch.full((len(x), T_MAX - x.shape[1]), PAD)], 1)


real = checks(padded(X)).float().mean(0)
for name, v in zip(CHECKS, real.tolist()):
    print(f"  {name:16} {pct(v, 2)}")
assert real.min() >= 0.99 and np.allclose(real.tolist(), RUNS["setup"]["real"], atol=1e-4), "khác với kết quả đã lưu"

Ba phép kiểm tra không đạt 100% vì vài chữ trong truyện được đọc theo lối xưa, hay vì luật vần viết tay chưa bắt hết
mọi trường hợp. Nhưng mức gần 100% cho thấy 7 phép kiểm tra đúng là luật mà Nguyễn Du theo.

### Mô hình SFT

Chính sách là một GPT rất nhỏ: 2 khối Transformer, $d = 64$, 4 đầu chú ý, lớp ra dùng chung ma trận với lớp nhúng
(như GPT-2 ở Bài 17). `sweep.py` đã huấn luyện nó trên 1.300 cặp câu huấn luyện, chọn bước có mất mát kiểm định thấp
nhất, rồi làm tròn trọng số về float16. Ô dưới nạp đúng các trọng số đó. Đây là $\pi_{\text{ref}}$ cho mọi thí nghiệm.

In [ ]:
D, HEADS = 64, 4


class Block(nn.Module):
    """Khối Transformer pre-LN: tự chú ý nhân quả, rồi MLP GELU, mỗi phần nằm trong một kết nối tắt."""

    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.qkv, self.proj = nn.Linear(D, 3 * D), nn.Linear(D, D)
        self.fc, self.out = nn.Linear(D, 4 * D), nn.Linear(4 * D, D)

    def forward(self, h):
        B, T, _ = h.shape
        q, k, v = self.qkv(self.ln1(h)).view(B, T, 3, HEADS, D // HEADS).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        h = h + self.proj(a.transpose(1, 2).reshape(B, T, D))
        return h + self.out(F.gelu(self.fc(self.ln2(h))))


class GPT(nn.Module):
    """GPT nhỏ trên âm tiết; lớp ra là ma trận nhúng. Không bao giờ sinh ra <pad> hay <sep>."""

    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.pos = nn.Parameter(torch.zeros(T_MAX, D))
        self.blocks = nn.ModuleList(Block() for _ in range(2))
        self.ln = nn.LayerNorm(D)

    def hidden(self, x):
        h = self.emb(x) + self.pos[: x.shape[1]]
        for b in self.blocks:
            h = b(h)
        return self.ln(h)

    def forward(self, x):
        z = self.hidden(x) @ self.emb.weight.T
        return z.index_fill(-1, torch.tensor([PAD, SEP]), -1e9)  # exp(−1e9) = 0 đúng bằng 0


def unpack(m, d):
    """Nạp trọng số float16 (base64) vào mô hình."""
    sd = m.state_dict()
    for k, v in d.items():
        a = np.frombuffer(base64.b64decode(v["f16"]), np.float16).astype(np.float32).reshape(v["shape"])
        sd[k] = torch.from_numpy(a.copy())
    m.load_state_dict(sd)
    return m


def frozen(m):
    m = copy.deepcopy(m)
    for p in m.parameters():
        p.requires_grad_(False)
    return m


sft = frozen(unpack(GPT(), W["sft"]))
n_params = sum(p.numel() for p in sft.parameters())
print(f"{vn(n_params, 0)} tham số; bước SFT được chọn: {int(RUNS['setup']['sft']['step'])}")
assert n_params == RUNS["setup"]["params"] == 254016

Ta lấy mẫu bằng **CDF ngược** từ các số ngẫu nhiên rút trước của một `torch.Generator` có seed: mỗi câu trả lời dùng
đúng 13 số, nên cùng seed luôn cho cùng câu, dù câu kết thúc sớm hay muộn. Hàm `evaluate` cho chính sách viết một câu
bát cho mỗi lời nhắc kiểm tra và đo: điểm gold, tỉ lệ đạt từng phép kiểm tra, độ lệch $\mathrm{KL}(\pi\,\|\,\pi_{\text{ref}})$
của câu (tính chính xác trên cả bộ từ vựng ở từng bước, rồi cộng dọc câu), độ dài, và tỉ lệ cặp âm tiết liền nhau khác
nhau (distinct-2: thấp nghĩa là câu lặp lại).

In [ ]:
def sample(model, prompts, gen, temp=1.0):
    """Câu trả lời sau mỗi lời nhắc (PLEN token), lấy mẫu bằng CDF ngược; đệm đến T_MAX."""
    seq, done = prompts, torch.zeros(len(prompts), dtype=torch.bool)
    u = torch.rand(RESP, len(prompts), 1, generator=gen)
    with torch.no_grad():
        for t in range(RESP):
            if done.all():
                seq = torch.cat([seq, torch.full((len(seq), RESP - t), PAD)], 1)
                break
            p = torch.softmax(model(seq)[:, -1] / temp, -1)
            nxt = (p.cumsum(-1) < u[t]).sum(-1).clamp(max=p.shape[-1] - 1)
            nxt = torch.where(done, PAD, nxt)
            done = done | (nxt == EOS)
            seq = torch.cat([seq, nxt[:, None]], 1)
    return seq


def resp_mask(seq):
    """Những đích seq[:, 1:] thuộc câu trả lời (âm tiết và <eos>, không phải phần đệm)."""
    tgt = seq[:, 1:]
    m = torch.zeros_like(tgt, dtype=torch.bool)
    m[:, PLEN - 1 :] = True
    return m & (tgt != PAD)


def token_logp(model, seq):
    """log π(token | phần trước) của mọi đích seq[:, 1:], và cả phân phối log."""
    lsm = torch.log_softmax(model(seq[:, :-1]), -1)
    return lsm.gather(-1, seq[:, 1:, None]).squeeze(-1), lsm


def seq_logp(model, seq):
    lp, _ = token_logp(model, seq)
    return (lp * resp_mask(seq)).sum(1)


def exact_kl(model, ref, seq):
    """KL(π‖π_ref) của từng câu đã lấy mẫu: chính xác trên bộ từ vựng ở từng bước, cộng dọc câu."""
    with torch.no_grad():
        _, a = token_logp(model, seq)
        _, b = token_logp(ref, seq)
        per = (a.exp() * (a - b)).sum(-1)
    return (per * resp_mask(seq)).sum(1)


def decode(seq, k=None):
    out = []
    for row in seq[:k].tolist():
        resp = []
        for t in row[PLEN:]:
            if t in (EOS, PAD):
                break
            resp.append(VOCAB[t])
        out.append(" ".join(VOCAB[t] for t in row[: PLEN - 1]) + " / " + " ".join(resp))
    return out


def evaluate(pol, rewards=None, copies=1):
    """Đo trên mẫu mới của chính sách cho 164 lời nhắc kiểm tra (luôn cùng một dòng số ngẫu nhiên, seed 2000)."""
    seq = sample(pol, X_test[:, :PLEN].repeat(copies, 1), torch.Generator().manual_seed(2000))
    c, n, r = checks(seq).float(), resp_len(seq), seq[:, PLEN:]
    pair = r[:, :-1] * V + r[:, 1:]
    bg = pair[torch.arange(RESP - 1)[None] < (n - 1)[:, None]]
    out = {"gold": c.mean().item(), "checks": c.mean(0).tolist(), "kl": exact_kl(pol, sft, seq).mean().item(),
           "len": n.float().mean().item(), "distinct2": len(set(bg.tolist())) / max(1, len(bg)), "top": []}  # fmt: skip
    for j in (1, 5, 7):  # âm tiết thường gặp nhất ở chữ 2, 6, 8 của câu trả lời, và tỉ lệ của nó
        v, cnt = torch.unique(r[:, j], return_counts=True)
        out["top"].append([VOCAB[v[cnt.argmax()]], cnt.max().item() / len(r)])
    for k, f in (rewards or {}).items():
        out[k] = f(seq).mean().item()
    return out, seq


ev_sft, seq_sft = evaluate(sft)
print("\n".join(decode(seq_sft, 6)))
print(
    f"\ngold {vn(ev_sft['gold'], 3)} · độ dài {vn(ev_sft['len'], 2)} âm tiết · distinct-2 {vn(ev_sft['distinct2'], 3)}"
)
for name, v in zip(CHECKS, ev_sft["checks"]):
    print(f"  {name:16} {pct(v)}")
saved = RUNS["live"]["check"]["sft"]
assert abs(ev_sft["gold"] - saved["gold"]) < 0.01, "khác với kết quả đã lưu"

Câu của $\pi_{\text{ref}}$ có đúng dáng câu bát: 95,7% có 8 âm tiết, và chữ 2, 6, 8 mang thanh bằng ở 84,8–86,6% số câu,
vì đó là những điều lặp lại trong mọi câu bát của truyện. Nhưng câu không có nghĩa: một mô hình 254.016 tham số học từ
1.300 câu chỉ nắm được thống kê bề mặt. Hai phép kiểm tra cần *so* hai chữ với nhau thì hiếm khi đạt: chữ 6 vần với lời
nhắc ở 12,8% số câu, cặp huyền/ngang ở chữ 6–8 ở 36,0%. Điểm gold 0,679 là chỗ mọi thí nghiệm bên dưới bắt đầu.

### Người chấm mô phỏng

Mỗi lần so sánh, ta cho $\pi_{\text{ref}}$ viết hai câu bát cho cùng một lời nhắc. Người chấm mô phỏng chọn theo
Bradley–Terry trên điểm gold, với độ "tinh mắt" $s = 10$:
$$P(y_a \succ y_b) = \sigma\big(s\,(g_a - g_b)\big).$$
Hai câu hơn nhau một phép kiểm tra ($\Delta g = 1/7$) thì câu tốt hơn được chọn với xác suất $\sigma(10/7)$. Gao,
Schulman & Hilton (2022) cũng thay người bằng một "mô hình phần thưởng gold" như vậy để đo Goodhart, nhưng nhãn của họ
không ồn (luôn chọn câu có điểm gold cao hơn). Ta chọn $s$ sao cho hai người chấm độc lập đồng ý với nhau khoảng như
người chấm của InstructGPT. Ô dưới chấm lại 1.000 cặp huấn luyện đã lưu, bằng đúng dòng số ngẫu nhiên của script.

In [ ]:
S = RUNS["setup"]["S"]


def unpack_pairs(d):
    a = torch.from_numpy(np.frombuffer(base64.b64decode(d["a"]), np.int16).astype(np.int64).reshape(-1, T_MAX))
    b = torch.from_numpy(np.frombuffer(base64.b64decode(d["b"]), np.int16).astype(np.int64).reshape(-1, T_MAX))
    return a, b, torch.tensor(d["y"], dtype=torch.float32)


def annotate(a, b, seed, split):
    """1 khi người chấm chọn a: P(a ≻ b) = σ(S·(g_a − g_b))."""
    u = torch.rand(len(a), generator=torch.Generator().manual_seed(15_000 + 1000 * split + seed))
    return (u < torch.sigmoid(S * (gold(a) - gold(b)))).float()


pairs = {sp: unpack_pairs(W["pairs"][sp]) for sp in ("train", "val", "test")}
a_tr, b_tr, y_tr = pairs["train"]
assert torch.equal(annotate(a_tr, b_tr, 0, 0), y_tr), "nhãn khác với tệp đã lưu"
ga, gb = gold(a_tr), gold(b_tr)
p = torch.sigmoid(S * (ga - gb))
print("một cặp:", decode(a_tr[:1])[0], f"(gold {vn(ga[0], 3)})")
print("        ", decode(b_tr[:1])[0], f"(gold {vn(gb[0], 3)})")
print(f"P(chọn câu đầu) = {vn(p[0], 3)}; người chấm đã chọn câu {'đầu' if y_tr[0] else 'sau'}")
print(f"\n1.000 cặp: hòa điểm gold {pct((ga == gb).float().mean())}, "
      f"hai người chấm độc lập đồng ý {pct((p**2 + (1 - p) ** 2).mean())}")  # fmt: skip

### 🤔 Dự đoán trước
Giả sử một mô hình phần thưởng *hoàn hảo*: nó biết đúng điểm gold của mọi câu và luôn chọn câu có $g$ cao hơn (hòa
thì đoán bừa). Trên các cặp kiểm tra, nó đoán đúng nhãn của người chấm bao nhiêu phần trăm? 100%? 90%? Ít hơn?

In [ ]:
a_te, b_te, y_te = pairs["test"]
ga_t, gb_t = gold(a_te), gold(b_te)
p_t = torch.sigmoid(S * (ga_t - gb_t))
perfect = torch.where(ga_t == gb_t, 0.5, ((ga_t > gb_t).float() == y_te).float())
print(f"mô hình hoàn hảo đoán đúng {pct(perfect.mean())} nhãn của 1.000 cặp kiểm tra")
print(
    f"trần lý thuyết E[max(p, 1 − p)]: {pct(torch.maximum(p_t, 1 - p_t).mean())} "
    f"(trên cả 2.000 cặp kiểm tra của script: {pct(RUNS['setup']['ceiling'])})"
)

Không phải 100%, mà 79,4%. Nhãn của người chấm vốn ồn: ở cặp ví dụ trên, hai câu hơn nhau một phép kiểm tra, và câu tốt
hơn chỉ được chọn với xác suất 0,807. Khi hai câu hòa điểm gold (21,6% số cặp huấn luyện), nhãn chỉ là tung đồng xu. Về
lâu dài, không mô hình nào đoán nhãn tốt hơn trần $\mathbb{E}[\max(p, 1-p)]$, 78,3% trên 1.000 cặp này, kể cả mô hình biết
hết luật; 79,4% cao hơn trần một chút chỉ vì may trên một mẫu hữu hạn. Hai người chấm độc lập cũng chỉ đồng ý với nhau
73,2%, trong khoảng 72,6–77,3% của InstructGPT. Vì thế độ chính xác của một mô hình phần thưởng phải đọc so với trần này,
không so với 100%.

## 6 · Mô hình phần thưởng

Mô hình phần thưởng là bản sao của mô hình SFT, bỏ lớp ra và thay bằng một đầu tuyến tính cho ra một số: điểm của cả
câu, đọc ở token cuối (`<eos>`). Lớp nhúng giữ nguyên như SFT (đóng băng), giống InstructGPT khởi tạo mô hình phần
thưởng từ mô hình SFT. Mất mát là Bradley–Terry của mục 4 trên hiệu điểm của hai câu. Ô dưới huấn luyện nó 2 lượt trên
1.000 cặp, chọn lượt có mất mát thấp nhất trên 500 cặp kiểm định (theo nhãn của người chấm, không bao giờ theo gold),
rồi đo trên 1.000 cặp kiểm tra.

In [ ]:
class Scorer(nn.Module):
    """Mô hình phần thưởng / mô hình giá trị: bản sao thân của mô hình SFT (lớp nhúng đóng băng) và một đầu ra vô hướng."""

    def __init__(self, base):
        super().__init__()
        self.body = copy.deepcopy(base)
        for p_ in self.body.parameters():
            p_.requires_grad_(True)
        self.body.emb.weight.requires_grad_(False)
        self.head = nn.Linear(D, 1)

    def values(self, seq):
        """Một giá trị ở mỗi vị trí, thẳng hàng với các đích seq[:, 1:] (dùng cho mô hình giá trị của PPO)."""
        return self.head(self.body.hidden(seq[:, :-1])).squeeze(-1)

    def forward(self, seq):
        """Điểm của cả câu: đầu ra ở token cuối không phải phần đệm (<eos>)."""
        h = self.body.hidden(seq)
        last = (seq != PAD).sum(1) - 1
        return self.head(h[torch.arange(len(seq)), last]).squeeze(-1)


def bt_loss(rm, a, b, y):
    return F.binary_cross_entropy_with_logits(rm(a) - rm(b), y)


def train_rm(a, b, y, va, seed, epochs, lr=1e-3, batch=64):
    torch.manual_seed(seed)
    rm = Scorer(sft)
    opt = torch.optim.Adam([p for p in rm.parameters() if p.requires_grad], lr)
    g = torch.Generator().manual_seed(20_000 + seed)
    best, curve = (float("inf"), None, 0), []
    for ep in range(1, epochs + 1):
        for j in torch.randperm(len(y), generator=g).split(batch):
            loss = bt_loss(rm, a[j], b[j], y[j])
            opt.zero_grad()
            loss.backward()
            opt.step()
        with torch.no_grad():
            vl = bt_loss(rm, *va).item()
        curve.append((ep, vl))
        if vl < best[0]:
            best = (vl, copy.deepcopy(rm.state_dict()), ep)
    rm.load_state_dict(best[1])
    with torch.no_grad():  # làm tròn về float16 như script
        for p in rm.parameters():
            p.copy_(p.half().float())
    return frozen(rm), curve


def rm_accuracy(rm, a, b, y):
    with torch.no_grad():
        d = rm(a) - rm(b)
    g_a, g_b = gold(a), gold(b)
    return ((d > 0).float() == y).float().mean().item(), ((d > 0) == (g_a > g_b))[g_a != g_b].float().mean().item()


rm_live, curve = train_rm(a_tr, b_tr, y_tr, pairs["val"], 0, RUNS["live"]["cfg"]["rmEpochs"])
acc, gacc = rm_accuracy(rm_live, a_te, b_te, y_te)
print("mất mát kiểm định theo lượt:", ", ".join(vn(v, 4) for _, v in curve))
print(f"đúng nhãn người chấm {pct(acc)} · đúng thứ tự gold (bỏ cặp hòa) {pct(gacc)}")
assert abs(acc - RUNS["live"]["rm"]["acc"]) < 0.01, "khác với kết quả đã lưu"

Sau 2 lượt trên 1.000 cặp, mô hình đoán đúng 61,2% nhãn (trần 78,3%) và đúng thứ tự gold ở 66,8% số cặp không hòa:
hơn hẳn đoán bừa, nhưng còn xa trần.

Script đã huấn luyện mô hình phần thưởng trên 250 đến 16.000 cặp, mỗi cỡ 3 seed, tối đa 8 lượt (lượt chọn trên cặp
kiểm định). Hình dưới vẽ độ chính xác trên 2.000 cặp kiểm tra theo số cặp: đúng nhãn của người chấm (tròn, xanh dương),
đúng thứ tự gold (vuông, cam), so với trần (nét gạch). Ô cũng in hệ số tương quan giữa điểm của mô hình và điểm gold
trên các câu của $\pi_{\text{ref}}$.

In [ ]:
NS = sorted({x["n"] for x in RUNS["rm"] if x["kind"] == "plain"})


def rm_rows(kind, n):
    return [x for x in RUNS["rm"] if x["kind"] == kind and x["n"] == n]


fig, ax = plt.subplots(figsize=(7, 3.6))
for key, label, color, marker in (("acc", "đúng nhãn người chấm", BLUE, "o"),
                                  ("goldAcc", "đúng thứ tự gold", ORANGE, "s")):  # fmt: skip
    v = np.array([[x[key] for x in rm_rows("plain", n)] for n in NS])
    ax.plot(NS, v.mean(1), color=color, marker=marker, label=label)
    ax.fill_between(NS, v.min(1), v.max(1), color=color, alpha=0.2)
ax.axhline(RUNS["setup"]["ceiling"], color=GREY, ls="--", label="trần của đúng nhãn (nhãn ồn)")
ax.set(xscale="log", xlabel="số cặp so sánh để huấn luyện", ylabel="độ chính xác trên cặp kiểm tra",
       title="Mô hình phần thưởng: càng nhiều so sánh càng đúng")  # fmt: skip
ax.legend(frameon=False)
ax.minorticks_off()
vn_axes(fig)
ax.set_xticks(NS, [vn(n, 0) for n in NS])  # sau vn_axes, để giữ vạch ở đúng bốn cỡ đã đo
plt.show()
for n in NS:
    rows = rm_rows("plain", n)
    print(
        f"{vn(n, 0):>6} cặp: đúng nhãn {pct(min(x['acc'] for x in rows))}–{pct(max(x['acc'] for x in rows))}, "
        f"tương quan với gold {vn(min(x['corr'] for x in rows), 2)}–{vn(max(x['corr'] for x in rows), 2)}"
    )

Độ chính xác tăng đều theo số cặp, từ 53,5–61,9% (250 cặp) đến 69,3–71,2% (16.000 cặp), và vẫn dưới trần 78,0%.
InstructGPT báo cáo 72,4% với nhãn của chính những người chấm đã cho dữ liệu huấn luyện, 69,6% với người chấm mới. Đường
cam vượt trần vì nó đo một thứ khác: thứ tự gold, không phải nhãn ồn. Mô hình phần thưởng đoán *câu nào hay hơn thật*
đúng hơn là đoán *người chấm đã chọn gì*. Tương quan với gold cũng tăng, từ 0,35–0,49 lên 0,74–0,76.

Từ đây, mọi thí nghiệm tối ưu theo mô hình phần thưởng **4.000 cặp** của seed 0, mà tệp trọng số có kèm. Nó nhìn
thấy điều gì trong một câu thơ? Ô dưới cho $\pi_{\text{ref}}$ viết 16 câu cho mỗi 64 lời nhắc kiểm tra, chấm cả 1.024
câu, rồi hồi quy tuyến tính điểm của mô hình phần thưởng theo 7 phép kiểm tra: mỗi hệ số là số điểm mà mô hình
"cộng thêm" khi một phép kiểm tra đạt.

In [ ]:
rm = frozen(unpack(Scorer(sft), W["rm"]))  # tệp không chứa lớp nhúng: nó đúng bằng lớp nhúng của SFT
cfg = RUNS["live"]["cfg"]
P_bon = X_test[: cfg["bonPrompts"], :PLEN]
seq_bon = sample(sft, P_bon.repeat_interleave(cfg["bonN"], 0), torch.Generator().manual_seed(cfg["bonSeed"]))
with torch.no_grad():
    raw_score = rm(seq_bon)
C = checks(seq_bon).float()
coef = torch.linalg.lstsq(torch.cat([C, torch.ones(len(C), 1)], 1), raw_score[:, None]).solution.squeeze()
for name, w, share in zip(CHECKS, coef.tolist(), C.mean(0).tolist()):
    print(f"  {name:16} {vn(w, 2):>6}   (đạt ở {pct(share)} số câu SFT)")
print("tương quan với gold:", vn(np.corrcoef(raw_score.numpy(), gold(seq_bon).numpy())[0, 1], 3))

Mô hình phần thưởng cộng 0,56 đến 1,10 điểm cho mỗi luật về độ dài và thanh, nhưng gần như 0 cho hai luật còn lại: vần
(0,03) và cặp huyền/ngang ở chữ 6–8 (−0,02). Hai luật này cần *so* hai chữ với nhau (chữ 6 với chữ 6 của lời nhắc, chữ 6
với chữ 8), và câu SFT hiếm khi đạt, nên 4.000 lần so sánh ồn chứa ít thông tin về chúng. Tương quan với gold vẫn khá cao
(0,674), vì năm luật dễ chiếm phần lớn chênh lệch giữa các câu SFT. Nhưng đây là một chỗ mù, và tối ưu sẽ tìm ra nó.

## 7 · Best-of-N và định luật Goodhart

Cách đơn giản nhất để "tối ưu theo" mô hình phần thưởng là không huấn luyện gì cả: viết $N$ câu rồi giữ câu được điểm
cao nhất. Best-of-N lệch khỏi $\pi_{\text{ref}}$ không quá $\log N - (N-1)/N$ nat (Bài 26 đã tính), nên ta có một trục
chung cho mọi cách tối ưu: **độ lệch KL**. Gao, Schulman & Hilton (2022) cho mô hình phần thưởng 6 tỉ tham số của
InstructGPT đóng vai "gold", huấn luyện các mô hình phần thưởng nhỏ hơn từ nhãn của nó, rồi tối ưu một chính sách
1,2 tỉ tham số theo chúng. Vẽ theo $d = \sqrt{\mathrm{KL}}$, điểm gold tăng rồi giảm, trong khi điểm của mô hình phần
thưởng học được (gọi là *proxy*) cứ tăng mãi. Họ khớp được hai dạng hàm cho phần tăng của gold: $d\,(a - b\,d)$ cho
best-of-N và $d\,(a - b \log d)$ cho học tăng cường (họ gọi các hệ số là α, β; ta đổi tên vì β ở đây là hệ số phạt KL).

### Trên một bảng, tính chính xác: đuôi của sai số

Vì sao gold lại *giảm*? Proxy bằng gold cộng sai số, nên tối ưu proxy là chọn những câu có gold cao *hoặc* có sai số
lớn. Điều gì thắng thế phụ thuộc vào hình dạng của sai số (Kwa, Thomas & Garriga-Alonso, 2024). Ô dưới lấy một lời
nhắc với 5.000 câu trả lời, $\pi_{\text{ref}}$ đều, gold $g \sim \mathcal{N}(0, 1)$, và hai mô hình phần thưởng
$r = g + \varepsilon$. Sai số của chúng có cùng cỡ điển hình nhưng khác **đuôi**: $\varepsilon$ chuẩn (đuôi nhẹ), và
$\varepsilon$ Student-t với ν = 1,5 (đuôi nặng: thỉnh thoảng sai rất xa). Với một bảng, mọi thứ tính chính xác được:
chính sách tối ưu có phạt KL $\pi_\beta \propto \pi_{\text{ref}}\, e^{r/\beta}$ (Bài 26), xác suất best-of-N chọn câu
xếp hạng $k$ (tăng dần theo $r$) là $(k/K)^N - ((k-1)/K)^N$, và KL của cả hai.

In [ ]:
K = 5000
rng = np.random.default_rng(0)
g_tab = rng.standard_normal(K)
ERRORS = {"đuôi nhẹ": rng.standard_normal(K), "đuôi nặng": 0.6 * rng.standard_t(1.5, K)}


def tilt(r, beta):
    z = r / beta  # π_ref đều: log π_ref là một hằng số
    q = np.exp(z - z.max())
    return q / q.sum()


def kl_unif(q):
    m = q > 0  # quy ước 0·log 0 = 0
    return float(np.sum(q[m] * np.log(q[m] * K)))


def best_of(r, N):
    q, k = np.zeros(K), np.arange(1, K + 1)
    q[np.argsort(r)] = (k / K) ** N - ((k - 1) / K) ** N
    return q


BETAS_TAB, NS_TAB = np.geomspace(100, 0.02, 40), 2 ** np.arange(13)
table = {}
for name, eps in ERRORS.items():
    r_tab = g_tab + eps
    table[name] = {}
    for how, qs in (("rl", [tilt(r_tab, b) for b in BETAS_TAB]), ("bon", [best_of(r_tab, N) for N in NS_TAB])):
        table[name][how] = [np.array([kl_unif(q) for q in qs]), np.array(qs) @ g_tab, np.array(qs) @ r_tab]
    (_, g_r, px_r), (_, g_b, px_b) = table[name]["rl"], table[name]["bon"]
    print(f"{name}: |ε| trung vị {vn(np.median(abs(eps)), 2)}, lớn nhất {vn(abs(eps).max(), 1)}")
    print(f"  best-of-N: gold cao nhất {vn(g_b.max(), 2)} ở N = {vn(NS_TAB[g_b.argmax()], 0)}; ở N = 4.096 gold "
          f"{vn(g_b[-1], 2)}, proxy {vn(px_b[-1], 1)}")  # fmt: skip
    print(f"  π_β: gold cao nhất {vn(g_r.max(), 2)} ở β = {vn(BETAS_TAB[g_r.argmax()], 2)}; ở β = 0,02 gold "
          f"{vn(g_r[-1], 2)}, proxy {vn(px_r[-1], 1)}")  # fmt: skip
assert abs(kl_unif(best_of(g_tab, 16)) - (np.log(16) - 15 / 16)) < 0.01  # KL của best-of-N gần đúng bằng cận

Hình dưới vẽ điểm gold theo $\sqrt{\mathrm{KL}}$: vòng tròn xanh dương là $\pi_\beta$ (β giảm dần từ trái sang phải),
hình vuông cam là best-of-N ($N = 1$ đến 4.096), nét chấm là gold của $\pi_{\text{ref}}$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, name in zip(axes, ERRORS):
    for how, color, marker, label in (("rl", BLUE, "o", "π_β"), ("bon", ORANGE, "s", "best-of-N")):
        kl_, g_, _ = table[name][how]
        ax.plot(np.sqrt(np.maximum(kl_, 0)), g_, color=color, marker=marker, ms=4, label=label)
    ax.axhline(g_tab.mean(), color=GREY, ls=":", lw=1)
    ax.set(xlabel="√KL(π‖π_ref)", title=f"Sai số {name}")
axes[0].set_ylabel("điểm gold")
axes[0].legend(frameon=False)
fig.suptitle("Cùng một gold, hai mô hình phần thưởng: chỉ khác đuôi của sai số", y=1.02)
vn_axes(fig)
plt.show()

**Đuôi nhẹ:** gold tăng suốt (lên 2,73 với $\pi_\beta$ ở β = 0,02, và 2,59 với best-of-N ở N = 4.096), dù chậm hơn
proxy (4,8 và 4,6). Câu được chọn vì gold cao *và* vì sai số dương, nên proxy luôn phóng đại chất lượng của nó (lời
nguyền của kẻ thắng), nhưng tối ưu thêm vẫn có lợi.

**Đuôi nặng:** best-of-N tăng gold đến N = 16 (0,89) rồi giảm, xuống −0,91 ở N = 4.096, dưới cả $\pi_{\text{ref}}$,
trong khi proxy lên 118,5. $\pi_\beta$ còn tệ hơn: câu có sai số 156,3 hút gần hết xác suất ngay từ những β rất lớn, nên
gold cao nhất chỉ 0,01 (ở β = 51,94). Kwa và cộng sự chứng minh điều này tổng quát: khi sai số có đuôi nặng, có những
chính sách đạt proxy cao tùy ý mà gold không hơn chính sách gốc, với KL nhỏ tùy ý. Phạt KL không ngăn được điều đó; họ gọi
nó là *Goodhart thảm họa*.

Sai số của mô hình phần thưởng thật không phải nhiễu ngẫu nhiên. Mục 6 cho thấy nó bỏ qua hẳn hai luật, nên mọi câu đạt
năm luật dễ mà phạm hai luật kia đều được nó chấm cao hơn giá trị thật, một cách có hệ thống. Tối ưu sẽ tìm đến đúng
những câu ấy.

### Best-of-N trên thơ lục bát

Ô dưới dùng lại 1.024 câu SFT của mục 6 (16 câu cho mỗi 64 lời nhắc), chấm bằng mô hình phần thưởng 4.000 cặp, và tính
điểm gold kỳ vọng của best-of-N cho $N = 1$ đến 16 bằng công thức chính xác cho một kho mẫu hữu hạn (rút $N$ câu không
hoàn lại trong 16 câu, như Gao và cộng sự): câu xếp hạng $i$ (tăng dần) là câu tốt nhất của $N$ câu với xác suất
$\binom{i-1}{N-1} / \binom{16}{N}$. Điểm proxy ở đây đã chuẩn hóa (mục 8).

In [ ]:
def bon_curve(score, Ns, **metrics):
    """E[chỉ số của câu được điểm cao nhất trong N câu], N câu rút không hoàn lại từ n câu của mỗi lời nhắc."""
    n = score.shape[1]
    order = np.argsort(score, 1, kind="stable")
    cols = {"score": score, **metrics}
    sorted_ = {k: np.take_along_axis(v, order, 1) for k, v in cols.items()}
    out = []
    for N in Ns:
        w = np.array([math.comb(i - 1, N - 1) for i in range(1, n + 1)], float) / math.comb(n, N)
        row = {"N": int(N), "klBound": float(np.log(N) - (N - 1) / N)}
        row.update({k: float((v * w).sum(1).mean()) for k, v in sorted_.items()})
        out.append(row)
    return out


CHECK = RUNS["live"]["check"]
bon_live = bon_curve(((raw_score - CHECK["rmMu"]) / CHECK["rmSd"]).view(len(P_bon), -1).numpy(), [1, 2, 4, 8, 16],
                     gold=gold(seq_bon).view(len(P_bon), -1).numpy())  # fmt: skip
print(" N   KL ≤    proxy   gold")
for row in bon_live:
    print(f"{row['N']:2}   {vn(row['klBound'], 2)}   {vn(row['score'], 2):>6}   {vn(row['gold'], 3)}")
assert abs(bon_live[-1]["gold"] - RUNS["live"]["bon"][-1]["gold"]) < 0.01, "khác với kết quả đã lưu"

Chọn 1 trong 16 câu nâng gold từ 0,682 lên 0,797, với KL không quá 1,84 nat, mà không đổi một trọng số nào.

Script làm cùng phép tính với 256 câu cho mỗi 164 lời nhắc kiểm tra, cho từng mô hình phần thưởng (250 đến 16.000
cặp) và cho chính gold, 3 seed. Hình dưới vẽ gold theo $\sqrt{\mathrm{KL}}$ (dùng cận trên $\log N - (N-1)/N$, rất sát
với KL thật khi các câu ít trùng nhau): mỗi đường là một mô hình phần thưởng, dải màu là khoảng giữa các seed.

In [ ]:
def seeds_of(rows, key):
    return np.array([[e[key] for e in row] for row in rows])


fig, ax = plt.subplots(figsize=(7.5, 4))
styles = {"rm250": (GREY, "v", "250 cặp"), "rm1000": (GREEN, "^", "1.000 cặp"), "rm4000": (BLUE, "o", "4.000 cặp"),
          "rm16000": (ORANGE, "s", "16.000 cặp"), "gold": ("black", "*", "chính gold")}  # fmt: skip
for name, (color, marker, label) in styles.items():
    rows = [b["curves"][name] for b in RUNS["bon"]]
    d = np.sqrt([e["klBound"] for e in rows[0]])
    v = seeds_of(rows, "gold")
    ax.plot(d, v.mean(0), color=color, marker=marker, ms=5, label=label)
    ax.fill_between(d, v.min(0), v.max(0), color=color, alpha=0.15)
ax.set(
    xlabel="√KL (cận trên của best-of-N)",
    ylabel="điểm gold",
    title="Best-of-N trên 164 lời nhắc kiểm tra, N = 1 đến 256",
)
ax.legend(frameon=False, title="chọn theo", fontsize=8)
vn_axes(fig)
plt.show()
for name in styles:
    v = seeds_of([b["curves"][name] for b in RUNS["bon"]], "gold")
    k = v.mean(0).argmax()
    print(
        f"{styles[name][2]:>11}: N = 1: {vn(v[:, 0].mean(), 3)} · cao nhất (trung bình seed) ở N = "
        f"{int(RUNS['bon'][0]['curves'][name][k]['N'])}: {vn(v[:, k].min(), 3)}–{vn(v[:, k].max(), 3)}"
        f" · N = 256: {vn(v[:, -1].min(), 3)}–{vn(v[:, -1].max(), 3)}"
    )

Với mô hình phần thưởng 4.000 cặp, gold đạt 0,788–0,796 ở N = 64 rồi đứng yên đến N = 256 (0,783–0,798). Ở mức tối ưu
nhẹ này, với mọi mô hình phần thưởng, gold chưa giảm rõ hơn dao động giữa các seed. Mô hình càng học nhiều cặp, đường càng
cao: với 16.000 cặp, gold lên 0,789–0,823 ở N = 256. Chọn theo chính gold thì lên 0,996. Khoảng cách giữa đường đen và các
đường màu là cái giá của việc chỉ có một mô hình phần thưởng gần đúng.

## 8 · PPO từng mảnh

Best-of-N tốn $N$ lần viết cho mỗi câu trả lời. Học tăng cường thì đưa sự tối ưu vào chính trọng số. InstructGPT dùng
PPO (Schulman và cộng sự, 2017) với các thành phần sau, ta lắp từng cái.

**Phần thưởng chuẩn hóa.** Theo mục 4, thang đo của mô hình phần thưởng là tùy ý. Ta trừ trung bình và chia độ lệch
chuẩn của nó trên các câu SFT viết cho lời nhắc kiểm định, nên "1" nghĩa là hơn câu SFT trung bình một độ lệch chuẩn,
với mọi mô hình phần thưởng (và với gold).

In [ ]:
class Reward:
    """Một hàm phần thưởng trên câu, chuẩn hóa trên câu SFT cho các lời nhắc kiểm định (4 câu mỗi lời nhắc)."""

    def __init__(self, fn, seed=0):
        self.fn = fn
        with torch.no_grad():
            s = fn(sample(sft, X_val[:, :PLEN].repeat(4, 1), torch.Generator().manual_seed(30_000 + seed)))
        self.mu, self.sd = s.mean().item(), s.std().item()

    def __call__(self, seq):
        with torch.no_grad():
            return (self.fn(seq) - self.mu) / self.sd


R = Reward(rm)
print(f"điểm thô của mô hình phần thưởng trên câu SFT: trung bình {vn(R.mu, 3)}, độ lệch chuẩn {vn(R.sd, 3)}")
assert abs(R.mu - CHECK["rmMu"]) < 1e-3 and abs(R.sd - CHECK["rmSd"]) < 1e-3, "khác với kết quả đã lưu"

**Phần thưởng theo từng token.** Mỗi câu trả lời là một chuỗi hành động (từng âm tiết). Phạt KL được trả ở *mọi*
bước, còn điểm của mô hình phần thưởng chỉ đến ở token cuối:
$$r_t = -\beta\,\big(\log\pi(y_t \mid \cdot) - \log\pi_{\text{ref}}(y_t \mid \cdot)\big) + [\,t = \text{cuối}\,]\cdot r_\phi(x, y).$$
Cộng dọc câu, phần phạt là $\beta$ nhân ước lượng một mẫu của $\mathrm{KL}(\pi\,\|\,\pi_{\text{ref}})$.

**Mô hình giá trị và GAE.** Một mô hình thứ hai, $V(s_t)$, đoán tổng phần thưởng còn lại từ bước $t$; InstructGPT khởi
tạo nó từ mô hình phần thưởng, ta cũng vậy. Lợi thế (advantage) của từng token theo GAE (Schulman và cộng sự, 2016),
với $\gamma = 1$ và $\lambda = 0{,}95$:
$$\delta_t = r_t + V(s_{t+1}) - V(s_t), \qquad \hat A_t = \delta_t + \lambda\,\hat A_{t+1}.$$
$\lambda = 1$ cho lợi thế Monte Carlo (ít lệch, nhiều ồn); $\lambda = 0$ chỉ tin $V$ (ít ồn, lệch theo sai số của $V$).
Ô dưới tính tất cả cho một câu của một chính sách thử, hơi khác $\pi_{\text{ref}}$ để phần phạt KL khác 0, và in
từng token.

In [ ]:
def critic_from(scorer, reward):
    """Mô hình giá trị khởi đầu là mô hình phần thưởng, đổi thang theo phần thưởng chuẩn hóa."""
    c = copy.deepcopy(scorer)
    with torch.no_grad():
        c.head.weight /= reward.sd
        c.head.bias.sub_(reward.mu).div_(reward.sd)
    return c


def rollout(pol, crit, reward, beta, prompts, g):
    seq = sample(pol, prompts, g)
    m = resp_mask(seq).float()
    with torch.no_grad():
        lp_old, _ = token_logp(pol, seq)
        lp_ref, _ = token_logp(sft, seq)
        v_old = crit.values(seq) * m
        rew = -beta * (lp_old - lp_ref) * m
        last = (m * torch.arange(m.shape[1])).argmax(1)
        score = reward(seq)
        rew[torch.arange(len(seq)), last] += score
    return seq, m, lp_old, lp_ref, v_old, rew, score


def gae(rew, v, m, lam=0.95):
    """Lợi thế GAE, γ = 1; sau token cuối không còn gì (không ước lượng tiếp bằng V)."""
    adv, nxt_v, nxt_a = torch.zeros_like(rew), torch.zeros(len(rew)), torch.zeros(len(rew))
    for t in reversed(range(m.shape[1])):
        delta = rew[:, t] + nxt_v - v[:, t]
        nxt_a = (delta + lam * nxt_a) * m[:, t]
        adv[:, t], nxt_v = nxt_a, v[:, t]
    return adv


crit0 = critic_from(rm, R)
demo_pol = copy.deepcopy(sft)  # một chính sách hơi khác π_ref để phần phạt KL khác 0
with torch.no_grad():
    demo_pol.blocks[1].out.bias.add_(0.05 * torch.randn(D, generator=torch.Generator().manual_seed(1)))
seq, m, lp_old, lp_ref, v_old, rew, score = rollout(demo_pol, crit0, R, 0.1, X_test[:1, :PLEN],
                                                    torch.Generator().manual_seed(5))  # fmt: skip
adv = gae(rew, v_old, m)
print(decode(seq)[0], f"   (điểm chuẩn hóa {vn(score[0], 3)})")
print("token      log π − log π_ref   r_t      V(s_t)   Â_t")
for t in torch.nonzero(m[0]).squeeze(1).tolist():
    tok = VOCAB[seq[0, t + 1]]
    print(
        f"{tok:9} {vn(lp_old[0, t] - lp_ref[0, t], 3):>12}   {vn(rew[0, t], 3):>7}   {vn(v_old[0, t], 3):>6}  "
        f"{vn(adv[0, t], 3):>6}"
    )

Đọc bảng. Chính sách thử chỉ khác $\pi_{\text{ref}}$ một chút, nên phần phạt KL trong $r_t$ gần như bằng 0, và phần
thưởng đến gần hết ở `<eos>` (0,228). Cột $V(s_t)$ thì lạ: từ −6,250 đến −1,298, trong khi cả câu chỉ được 0,228. Mô hình
giá trị khởi đầu từ mô hình phần thưởng, mà mô hình phần thưởng chỉ học chấm ở token `<eos>`; ở các vị trí trước đó, đầu
ra của nó chưa có nghĩa. Lúc đầu, lợi thế $\hat A_t$ vì thế chủ yếu đo sai số của $V$. Mô hình giá trị học dần trong các
vòng PPO, và việc chuẩn hóa lợi thế trên cả vòng bớt đi phần lệch chung.

**Tỉ số bị kẹp.** Mỗi vòng, PPO lấy 128 câu từ chính sách hiện tại rồi dùng chúng cho 4 lượt cập nhật (mỗi lượt 4 lô
32 câu). Sau lượt đầu, chính sách đã khác chính sách lấy mẫu, nên mục tiêu dùng tỉ số
$\rho_t = \pi(y_t)/\pi_{\text{old}}(y_t)$ và kẹp nó trong $[1 - \varepsilon, 1 + \varepsilon]$, $\varepsilon = 0{,}2$:
$$\mathcal{L}^{\text{clip}} = -\,\mathbb{E}_t\Big[\min\big(\rho_t \hat A_t,\; \mathrm{clip}(\rho_t, 1-\varepsilon, 1+\varepsilon)\,\hat A_t\big)\Big].$$
Khi $\rho_t = 1$, gradient của nó đúng bằng gradient REINFORCE của Bài 26. Mô hình giá trị học bằng sai số bình phương
với $\hat A_t + V_{\text{old}}(s_t)$. Lợi thế được chuẩn hóa trên mọi token của vòng. Ta theo dõi hai chỉ số: tỉ lệ token
bị kẹp, và KL xấp xỉ giữa chính sách cũ và mới; cả hai cho biết tốc độ học η có quá lớn không.

Ô dưới là toàn bộ vòng lặp. Nó chạy 8 vòng (64 câu mỗi vòng thay vì 128 cho nhanh), β = 0,01, trên mô hình phần thưởng
4.000 cặp, và in thống kê của chính các câu đã lấy mẫu ở mỗi vòng.

In [ ]:
def ppo(reward, critic0, beta, seed, iters, lr=3e-4, lr_v=1e-3, prompts=128, epochs=4, minibatch=32, clip=0.2):
    torch.manual_seed(seed)
    pol, crit = copy.deepcopy(sft), copy.deepcopy(critic0)
    for p_ in (*pol.parameters(), *crit.parameters()):
        p_.requires_grad_(True)
    crit.body.emb.weight.requires_grad_(False)
    opt_p = torch.optim.Adam(pol.parameters(), lr)
    opt_v = torch.optim.Adam([p_ for p_ in crit.parameters() if p_.requires_grad], lr_v)
    g = torch.Generator().manual_seed(40_000 + seed)
    trace = []
    for it in range(1, iters + 1):
        P = X_train[torch.randint(0, len(X_train), (prompts,), generator=g), :PLEN]
        seq, m, lp_old, lp_ref, v_old, rew, score = rollout(pol, crit, reward, beta, P, g)
        adv = gae(rew, v_old, m)
        ret = adv + v_old
        a = adv[m.bool()]
        adv = (adv - a.mean()) / (a.std() + 1e-8) * m
        stats = []
        for _ in range(epochs):
            for j in torch.randperm(len(seq), generator=g).split(minibatch):
                lp, _ = token_logp(pol, seq[j])
                ratio = torch.exp(lp - lp_old[j])
                mj = m[j]
                obj = torch.min(ratio * adv[j], torch.clamp(ratio, 1 - clip, 1 + clip) * adv[j])
                loss_p = -(obj * mj).sum() / mj.sum()
                opt_p.zero_grad()
                loss_p.backward()
                nn.utils.clip_grad_norm_(pol.parameters(), 1.0)
                opt_p.step()
                loss_v = (((crit.values(seq[j]) - ret[j]) * mj) ** 2).sum() / mj.sum()
                opt_v.zero_grad()
                loss_v.backward()
                nn.utils.clip_grad_norm_(crit.parameters(), 1.0)
                opt_v.step()
                with torch.no_grad():
                    lr_ = (lp - lp_old[j]).detach()
                    stats.append([(((ratio - 1).abs() > clip).float() * mj).sum().item() / mj.sum().item(),
                                  (((lr_.exp() - 1) - lr_) * mj).sum().item() / mj.sum().item()])  # fmt: skip
        clipfrac, approxkl = np.mean(stats, 0)
        trace.append({"it": it, "score": score.mean().item(), "gold": gold(seq).mean().item(),
                      "kl": ((lp_old - lp_ref) * m).sum(1).mean().item(), "clipfrac": clipfrac, "approxkl": approxkl})  # fmt: skip
    return pol, trace


pol_live, trace = ppo(R, crit0, cfg["beta"], 0, cfg["ppoIters"], prompts=cfg["ppoPrompts"])
print("vòng   proxy   gold    KL của câu   bị kẹp   KL xấp xỉ (cũ→mới)")
for e in trace:
    print(
        f"{e['it']:4}   {vn(e['score'], 2):>5}   {vn(e['gold'], 3)}   {vn(e['kl'], 2):>9}   {pct(e['clipfrac']):>6}"
        f"   {vn(e['approxkl'], 4)}"
    )
assert all(abs(e["gold"] - s["gold"]) < 0.03 for e, s in zip(trace, RUNS["live"]["ppo"])), "khác với kết quả đã lưu"

Trong 8 vòng, điểm proxy của chính các câu vừa lấy mẫu tăng từ −0,04 (vòng 1) lên 0,04–0,30, gold của chúng từ 0,665
lên 0,692–0,721, và KL của câu từ 0 lên 0,65 nat. Mỗi vòng chỉ có 64 câu nên các con số này ồn. Tỉ lệ token bị kẹp là
9,1% và 6,7% ở hai vòng đầu rồi 1,3–3,0%, và KL xấp xỉ giữa chính sách cũ và mới dưới 0,01: mỗi vòng chỉ đổi chính sách một chút,
đúng điều PPO muốn. Script chọn tốc độ học trước khi chạy, theo hai chỉ số này, không theo gold.

### Đặt KL vào đâu?

Có hai cách đưa $\beta\,\mathrm{KL}$ vào code: cộng vào **phần thưởng** (như trên, và như InstructGPT), hoặc cộng vào
**hàm mất mát** rồi để autograd lấy đạo hàm (như GRPO và nhiều thư viện). Cách thứ hai cần một ước lượng KL từ mẫu.
Hai ước lượng phổ biến, với $\rho = \pi_{\text{ref}}(y)/\pi(y)$ và $y \sim \pi$:
$k_1 = -\log\rho$ (không lệch, có thể âm) và $k_3 = \rho - 1 - \log\rho$ (không lệch, luôn $\geq 0$).
Cả hai đều *ước lượng* đúng giá trị KL. Câu hỏi là *đạo hàm* của chúng (Tang & Munos, 2025).

Thử trên một bài toán bandit: 100 hành động, phần thưởng $r$ ngẫu nhiên trong $[0, 1]$, β = 0,1, chính sách là softmax
của 100 logit khởi đầu tại $\pi_{\text{ref}}$; mỗi bước lấy 256 mẫu và cập nhật bằng Adam. Đích đúng là
$\pi^* \propto \pi_{\text{ref}}\,e^{r/\beta}$.

### 🤔 Dự đoán trước
Hàm mất mát là REINFORCE cộng $\beta \cdot \text{mean}(k_1)$ trên các mẫu. Sau 2.000 bước, chính sách có gần $\pi^*$
không? Nếu không, nó đi về đâu?

In [ ]:
torch.manual_seed(0)
logit_ref, r_band = torch.randn(100), torch.rand(100)
lp_ref_b = torch.log_softmax(logit_ref, 0)
beta_b = 0.1
star = torch.softmax(lp_ref_b + r_band / beta_b, 0)


def pi_rev(beta):
    """π_rev = β·π_ref/(λ − r), với λ > max r chọn bằng chia đôi sao cho tổng bằng 1."""
    lo, hi = r_band.max().item() + 1e-12, r_band.max().item() + 10
    for _ in range(200):
        lam_ = (lo + hi) / 2
        lo, hi = (lam_, hi) if (beta * lp_ref_b.exp() / (lam_ - r_band)).sum() > 1 else (lo, lam_)
    rev_ = beta * lp_ref_b.exp() / (lam_ - r_band)
    return rev_ / rev_.sum()


rev = pi_rev(beta_b)


def bandit(kind, steps=2000, batch=256, lr=0.05):
    g = torch.Generator().manual_seed(1)
    th = lp_ref_b.clone().requires_grad_(True)
    opt = torch.optim.Adam([th], lr)
    for _ in range(steps):
        lp = torch.log_softmax(th, 0)
        a = (lp.exp().cumsum(0)[None] < torch.rand(batch, 1, generator=g)).sum(1).clamp(max=99)
        log_ratio = lp[a] - lp_ref_b[a]  # log π(a) − log π_ref(a) = −log ρ
        if kind == "trong phần thưởng":
            Rk = (r_band[a] - beta_b * log_ratio).detach()
            loss = -((Rk - Rk.mean()) * lp[a]).mean()
        else:
            loss = -((r_band[a] - r_band[a].mean()) * lp[a]).mean()
            rho = torch.exp(-log_ratio)
            loss = loss + beta_b * (log_ratio if kind == "k1 trong mất mát" else rho - 1 - torch.log(rho)).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return torch.softmax(th.detach(), 0)


tv = lambda p_, q_: 0.5 * (p_ - q_).abs().sum().item()
band = {k: bandit(k) for k in ("trong phần thưởng", "k1 trong mất mát", "k3 trong mất mát")}
print(f"π_ref cách π* {vn(tv(lp_ref_b.exp(), star), 3)} (khoảng cách biến phân toàn phần, TV)")
for k, q in band.items():
    print(
        f"KL {k:18}: TV đến π* {vn(tv(q, star), 3)} · đến π_rev {vn(tv(q, rev), 3)} · xác suất lớn nhất {pct(q.max())}"
    )

**KL trong phần thưởng:** khoảng cách đến π* là 0,000. Tại π*, phần thưởng đã trừ phạt KL,
$r - \beta\log(\pi^*/\pi_{\text{ref}})$, bằng nhau ở mọi hành động, nên lợi thế bằng 0 trên từng mẫu và gradient tắt hẳn.

**$k_1$ trong mất mát:** chính sách dồn 99,8% xác suất vào một hành động, như thể không có phạt KL. Gradient của
$k_1 = \log\pi(a) - \log\pi_{\text{ref}}(a)$ theo tham số là $\nabla\log\pi(a)$, và kỳ vọng của nó khi $a \sim \pi$ bằng 0
(vì $\sum_a \nabla\pi(a) = 0$). Ước lượng thì đúng, nhưng đạo hàm của nó không phải đạo hàm của KL.

**$k_3$ trong mất mát:** chính sách hội tụ, nhưng về $\pi_{\text{rev}}$ (khoảng cách 0,024), không phải π* (0,520). Lấy kỳ
vọng của đạo hàm $(1-\rho)\nabla\log\pi(a)$ ta được $\nabla\,\mathrm{KL}(\pi_{\text{ref}}\,\|\,\pi)$: KL theo chiều ngược.
Cực đại $\mathbb{E}_\pi[r] - \beta\,\mathrm{KL}(\pi_{\text{ref}}\|\pi)$ cho $\pi_{\text{rev}} = \beta\,\pi_{\text{ref}}/(\lambda - r)$,
với λ chọn để tổng bằng 1; ô trên tìm λ bằng chia đôi. GRPO của DeepSeekMath (Shao và cộng sự, 2024) cộng đúng $k_3$ vào
hàm mất mát, nên thực ra nó phạt KL theo chiều này. Bài 26 thì đặt phạt KL vào phần thưởng, như cách thứ nhất. (Chuyên đề
03 cũng gặp chiều $\mathrm{KL}(\pi_0\,\|\,\pi)$: RL's Razor đo mức quên bằng nó.)

## 9 · PPO chạy lâu: tối ưu quá tay và lách luật

Script chạy PPO đầy đủ (128 câu mỗi vòng, 200 vòng, 3 seed) từ cùng $\pi_{\text{ref}}$ và đo trên 164 lời nhắc kiểm tra
(4 câu mỗi lời nhắc) sau mỗi 10 vòng. Trước hết, theo mô hình phần thưởng 4.000 cặp với β từ 0 đến 0,3.

### 🤔 Dự đoán trước
Gao và cộng sự thấy phạt KL chỉ làm điểm gold dừng lại sớm hơn *trên cùng một đường* gold–KL, như dừng sớm, chứ không
nâng đường đó lên. Ở đây cũng vậy không? Nghĩa là: tại cùng một độ lệch KL, chính sách β = 0,1 có điểm gold cao hơn
chính sách β = 0 không?

In [ ]:
def rl_rows(**kw):
    return [x for x in RUNS["rl"] if all(x[k] == v for k, v in kw.items())]


BETAS = sorted({x["beta"] for x in rl_rows(algo="ppo", reward="rm", n=4000)})
colors = dict(zip(BETAS, ["black", ORANGE, GREEN, BLUE, GREY]))
markers = dict(zip(BETAS, ["x", "s", "^", "o", "D"]))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for beta in BETAS:
    rows = rl_rows(algo="ppo", reward="rm", n=4000, beta=beta)
    for row in rows:
        d = np.sqrt([max(e["kl"], 0) for e in row["log"]])
        axes[0].plot(d, [e["gold"] for e in row["log"]], color=colors[beta], marker=markers[beta], ms=3, lw=0.8,
                     label=f"β = {vn(beta, 2)}" if row["seed"] == 0 else None)  # fmt: skip
    its = [e["it"] for e in rows[0]["log"]]
    v = seeds_of([r_["log"] for r_ in rows], "proxy")
    axes[1].plot(its, v.mean(0), color=colors[beta], marker=markers[beta], ms=3)
    axes[1].fill_between(its, v.min(0), v.max(0), color=colors[beta], alpha=0.15)
axes[0].set(xlabel="√KL(π‖π_ref)", ylabel="điểm gold", title="Gold theo độ lệch (mỗi đường một seed)")
axes[1].set(xlabel="vòng PPO", ylabel="proxy (chuẩn hóa)", title="Điểm của mô hình phần thưởng (3 seed)")
axes[0].legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.show()
for beta in BETAS:
    rows = rl_rows(algo="ppo", reward="rm", n=4000, beta=beta)
    peak = [max(e["gold"] for e in r_["log"]) for r_ in rows]
    end = [r_["log"][-1] for r_ in rows]
    print(
        f"β = {vn(beta, 2):>4}: gold cao nhất {vn(min(peak), 3)}–{vn(max(peak), 3)}; ở vòng 200 gold "
        f"{vn(min(e['gold'] for e in end), 3)}–{vn(max(e['gold'] for e in end), 3)}, KL "
        f"{vn(min(e['kl'] for e in end), 1)}–{vn(max(e['kl'] for e in end), 1)}"
    )
print("\nGold tốt nhất đã đạt khi KL còn dưới một ngân sách (khoảng giữa 3 seed):")
for beta in BETAS:
    rows = rl_rows(algo="ppo", reward="rm", n=4000, beta=beta)
    best = {b: [max(e["gold"] for e in r_["log"] if e["kl"] <= b) for r_ in rows] for b in (2, 4, 16)}
    print(f"β = {vn(beta, 2):>4}: " + " · ".join(f"KL ≤ {b}: {vn(min(v), 3)}–{vn(max(v), 3)}" for b, v in best.items()))

Ở đây thì có. Với mọi ngân sách KL, β = 0,1 đạt gold cao hơn β = 0: khi KL ≤ 4, 0,787–0,798 so với 0,730–0,758, hai
khoảng không chồng nhau; khi KL ≤ 2, β = 0,3 đạt 0,775–0,790 còn β = 0 chỉ 0,725–0,737. Không có phạt, gold đạt đỉnh
(0,756–0,778) rồi đi xuống, còn 0,722–0,740 ở vòng 200 với KL 34,7–48,7, trong khi proxy (hình phải) vẫn tăng hoặc
đứng ở mức cao: đúng hình dạng Goodhart mà Gao và cộng sự đo. Có phạt, đường gold–√KL dốc hơn hẳn ngay từ đầu (hình
trái). Một giả thuyết, notebook không kiểm tra: phạt KL không chỉ chặn chính sách lại sớm hơn mà còn đổi *hướng* nó đi,
vì chính sách phải giữ phần lớn phân phối của $\pi_{\text{ref}}$ và chỉ dịch xác suất ở nơi phần thưởng chênh nhiều.

Gao và cộng sự thấy khác, và chính họ ghi rằng kết luận của họ nhạy với siêu tham số. Một mô hình nhỏ với một bộ siêu
tham số PPO không đủ để phân xử. Điều chắc chắn hơn là β và dừng sớm không thay thế được cho nhau nếu chưa đo.

### Mô hình phần thưởng lớn hơn, và RLOO

Hình dưới giữ β = 0,01 và đổi số cặp so sánh mà mô hình phần thưởng học (250 đến 16.000), cùng với RLOO (Ahmadian và
cộng sự, 2024): REINFORCE không có mô hình giá trị, đường cơ sở là trung bình của 3 câu khác cho cùng lời nhắc, KL tính
trên cả câu như Bài 26.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
specs = [({"algo": "ppo", "n": 250, "beta": 0.01}, GREY, "v", "PPO, 250 cặp"),
         ({"algo": "ppo", "n": 1000, "beta": 0.01}, GREEN, "^", "PPO, 1.000 cặp"),
         ({"algo": "ppo", "n": 4000, "beta": 0.01}, BLUE, "o", "PPO, 4.000 cặp"),
         ({"algo": "ppo", "n": 16000, "beta": 0.01}, ORANGE, "s", "PPO, 16.000 cặp"),
         ({"algo": "rloo", "n": 4000, "beta": 0.01}, "black", "x", "RLOO, 4.000 cặp")]  # fmt: skip
for kw, color, marker, label in specs:
    rows = rl_rows(reward="rm", **kw)
    v = seeds_of([r_["log"] for r_ in rows], "gold")
    its = [e["it"] for e in rows[0]["log"]]
    ax.plot(its, v.mean(0), color=color, marker=marker, ms=4, label=label)
    ax.fill_between(its, v.min(0), v.max(0), color=color, alpha=0.15)
    print(
        f"{label:16}: gold ở vòng 200 {vn(v[:, -1].min(), 3)}–{vn(v[:, -1].max(), 3)}, cao nhất (trung bình seed) "
        f"{vn(v.mean(0).max(), 3)} ở vòng {int(its[v.mean(0).argmax()])}"
    )
ax.axhline(ev_sft["gold"], color=GREY, ls=":", lw=1)
ax.set(xlabel="vòng", ylabel="điểm gold", title="β = 0,01: theo số cặp so sánh của mô hình phần thưởng")
ax.legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.show()

Mô hình phần thưởng học nhiều cặp hơn thì gold cao hơn: ở vòng 200, 0,783–0,858 với 16.000 cặp so với 0,565–0,799
với 250 cặp. Với 250 và 1.000 cặp, trung bình seed đạt đỉnh ngay ở vòng 20 (0,725 và 0,737): mô hình phần thưởng yếu
bị khai thác sớm. Gao và cộng sự thấy cùng chiều đó khi tăng dữ liệu hay kích thước mô hình phần thưởng. Khoảng giữa
các seed rộng, vì mỗi seed có tập cặp, mô hình phần thưởng và quá trình RL riêng.

RLOO đạt 0,696–0,772 ở vòng 200, chồng với PPO cùng 4.000 cặp (0,717–0,814): trong dao động giữa các seed. Ahmadian và
cộng sự thấy RLOO ngang hoặc hơn PPO trên mô hình lớn; ở đây chỉ có thể nói nó không kém rõ rệt, dù không cần mô hình
giá trị.

### Đọc những câu thơ "lách luật"

Tệp trọng số có kèm chính sách PPO β = 0 sau 200 vòng (seed 0). Ô dưới cho nó viết một câu cho mỗi lời nhắc kiểm tra.

In [ ]:
ppo_ship = frozen(unpack(GPT(), W["ppo"]))
ev_ppo, seq_ppo = evaluate(ppo_ship, {"proxy": R})
print("\n".join(decode(seq_ppo, 6)))
print(
    f"\ngold {vn(ev_ppo['gold'], 3)} (SFT {vn(ev_sft['gold'], 3)}) · proxy {vn(ev_ppo['proxy'], 2)} · KL "
    f"{vn(ev_ppo['kl'], 1)} nat · distinct-2 {vn(ev_ppo['distinct2'], 3)}"
)
for name, v, v0 in zip(CHECKS, ev_ppo["checks"], ev_sft["checks"]):
    print(f"  {name:16} {pct(v):>6}  (SFT {pct(v0)})")
print("âm tiết thường gặp nhất ở chữ 2, 6, 8:", ", ".join(f"“{w}” {pct(s)}" for w, s in ev_ppo["top"]))
saved = RUNS["live"]["check"]["ppo"]
assert abs(ev_ppo["gold"] - saved["gold"]) < 0.01 and abs(ev_ppo["kl"] / saved["kl"] - 1) < 0.02, (
    "khác với kết quả đã lưu"
)

Câu nào cũng gần như một câu: "một tra … nước … tra … tra". Chính sách đã tìm ra đúng chỗ mù ở mục 6. "Tra" mang
thanh ngang và đứng ở cả chữ 2, 6, 8, nên câu đạt năm luật mà mô hình phần thưởng thấy (100,0%), còn luật huyền/ngang
không bao giờ đạt (0,0%), vì chữ 6 và chữ 8 cùng là "tra". Vần đạt 18,3%, ở những lời nhắc tình cờ có chữ 6 vần với
"tra". Proxy là 1,79 (đơn vị: độ lệch chuẩn của điểm trên câu SFT), nhưng gold chỉ từ 0,679 lên 0,740. Với KL 43,4 nat
và distinct-2 0,018, chính sách gần như đã quên mọi câu khác.

Đây là lách luật (reward hacking) đọc được bằng mắt. Ở mô hình lớn, nó thường khó thấy hơn: câu trả lời dài dòng,
chiều theo ý người hỏi (sycophancy), code qua được bài kiểm tra mà không đúng. MacDiarmid và cộng sự (2025) thấy một mô
hình học lách luật trong môi trường lập trình thật rồi khái quát sang những hành vi lệch chuẩn khác.

Ô dưới kiểm tra rằng trọng số đã nạp đúng là trọng số của script: log-xác suất của 4 câu thơ thật (teacher forcing)
dưới $\pi_{\text{ref}}$ và dưới chính sách PPO, và điểm của mô hình phần thưởng.

In [ ]:
Xp = padded(X_test[:4])
with torch.no_grad():
    lp_s, lp_p, sc = seq_logp(sft, Xp), seq_logp(ppo_ship, Xp), rm(Xp)
for k in range(4):
    print(decode(Xp[k : k + 1])[0])
    print(
        f"    gold {vn(gold(Xp[k : k + 1]).item(), 3)} · log π_ref {vn(lp_s[k], 1)} · log π_PPO {vn(lp_p[k], 1)} · "
        f"điểm thô của mô hình phần thưởng {vn(sc[k], 2)}"
    )
for name, v in (("sftLogp", lp_s), ("ppoLogp", lp_p), ("rm", sc)):
    assert np.allclose(v.numpy(), CHECK[name], atol=1e-3), "trọng số khác với của script"

Cả bốn câu đều là thơ thật, đạt đủ 7 luật. Dưới chính sách PPO, log-xác suất của chúng thấp hơn dưới $\pi_{\text{ref}}$
khoảng 60 nat: chính sách không còn viết thơ thật nữa. Mô hình phần thưởng chấm ba câu sau 2,21–2,81 (trung bình trên câu
SFT là 1,836), nhưng chấm câu đầu −0,58, dù câu đó đúng mọi luật. Sai số của nó không chỉ là hai luật bị bỏ qua.

### Người chấm thích câu dài, và tổ hợp mô hình phần thưởng

Hai thí nghiệm nữa ở β = 0,01, mỗi cái 3 seed:
- **Người chấm thích câu dài.** Singhal và cộng sự (2023) thấy phần lớn "tiến bộ" của RLHF trên vài bộ dữ liệu đến từ
  câu trả lời dài hơn. Ta cho người chấm một thiên lệch nhỏ: $u = g + 0{,}1\cdot(\text{số âm tiết} - 8)$. Luật lục bát
  (gold) thì phạt mọi câu khác 8 chữ.
- **Tổ hợp.** Coste và cộng sự (2024) huấn luyện nhiều mô hình phần thưởng và tối ưu theo điểm *thấp nhất* của chúng
  (thận trọng) hoặc trung bình. Ta dùng 3 mô hình, mỗi cái học một mẫu bootstrap của 4.000 cặp (Christiano và cộng sự,
  2017).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
specs = [("rm", BLUE, "o", "mô hình phần thưởng thường"), ("length", ORANGE, "s", "người chấm thích câu dài"),
         ("ens-min", GREEN, "^", "tổ hợp, lấy min"), ("ens-mean", GREY, "D", "tổ hợp, lấy trung bình")]  # fmt: skip
for spec, color, marker, label in specs:
    rows = rl_rows(algo="ppo", reward=spec, n=4000, beta=0.01)
    its = [e["it"] for e in rows[0]["log"]]
    for ax, key in zip(axes, ("len", "gold")):
        v = seeds_of([r_["log"] for r_ in rows], key)
        ax.plot(its, v.mean(0), color=color, marker=marker, ms=4, label=label)
        ax.fill_between(its, v.min(0), v.max(0), color=color, alpha=0.15)
    end = [r_["log"][-1] for r_ in rows]
    print(
        f"{label:27}: vòng 200 dài {vn(min(e['len'] for e in end), 1)}–{vn(max(e['len'] for e in end), 1)} âm tiết, "
        f"gold {vn(min(e['gold'] for e in end), 3)}–{vn(max(e['gold'] for e in end), 3)}, "
        f"KL {vn(min(e['kl'] for e in end), 1)}–{vn(max(e['kl'] for e in end), 1)}"
    )
axes[0].set(xlabel="vòng", ylabel="số âm tiết trung bình", title="Độ dài câu trả lời")
axes[1].set(xlabel="vòng", ylabel="điểm gold", title="Điểm gold")
axes[0].legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.show()
print(f"người chấm thích câu dài đổi {pct(RUNS['setup']['lengthFlip'])} số nhãn so với người chấm thường")

Người chấm thích câu dài chỉ đổi 2,3% số nhãn, nhưng thế là đủ: mô hình phần thưởng học rằng dài hơn thì tốt hơn, và
PPO kéo mọi câu lên 13,0 âm tiết, độ dài tối đa mà chính sách được viết. Gold rơi xuống 0,502–0,637, dưới cả SFT
(0,679), vì câu bát cần đúng 8 âm tiết. Singhal và cộng sự còn thấy rằng RLHF với phần thưởng *chỉ* dựa trên độ dài
đã tái tạo được phần lớn cải thiện so với mô hình SFT.

Tổ hợp ba mô hình phần thưởng (Coste và cộng sự, 2024), lấy min hay trung bình, cho gold 0,732–0,831 và 0,729–0,818 ở
vòng 200, so với 0,717–0,814 của một mô hình: trong dao động giữa các seed. Có lẽ vì ba thành viên học cùng một loại
dữ liệu (bootstrap của cùng 4.000 cặp, cùng thân SFT) nên có chung chỗ mù: không thành viên nào thấy luật vần, và lấy
min của ba lỗi giống nhau không sửa được lỗi đó. Tổ hợp giúp khi các thành viên sai *khác nhau*.

## 10 · DPO và IPO: học thẳng từ các cặp

Bài 26 đã suy ra DPO: chính sách tối ưu $\pi_\beta \propto \pi_{\text{ref}}\,e^{r/\beta}$ cho phép viết ngược
$r = \beta \log(\pi_\beta/\pi_{\text{ref}}) + \text{hằng số}$, rồi thay vào mất mát Bradley–Terry:
$$\mathcal{L}_{\text{DPO}} = -\log\sigma(\beta\,h), \qquad
  h = \log\frac{\pi(y_w)}{\pi_{\text{ref}}(y_w)} - \log\frac{\pi(y_l)}{\pi_{\text{ref}}(y_l)}.$$
Không cần mô hình phần thưởng, không cần lấy mẫu trong lúc học. IPO (Azar và cộng sự, 2023) thay $-\log\sigma$ bằng
bình phương: $\mathcal{L}_{\text{IPO}} = \big(h - \tfrac{1}{2\tau}\big)^2$.

Khác biệt hiện rõ khi người chấm **không bao giờ đổi ý**. Ô dưới lấy 3 câu trả lời, $\pi_{\text{ref}}$ đều, và ba cặp
so sánh luôn cho cùng kết quả: câu 1 ≻ câu 2 ≻ câu 3. Ta tối ưu logit của chính sách bằng gradient descent.

In [ ]:
W_, L_ = np.array([0, 0, 1]), np.array([1, 2, 2])  # ba cặp: 1 ≻ 2, 1 ≻ 3, 2 ≻ 3 (đánh số từ 0)


def tabular(kind, coef, steps, lr=0.5):
    th = np.zeros(3)  # logit; π_ref đều nên log π/π_ref = logit − hằng số, và h không phụ thuộc hằng số
    for _ in range(steps):
        h = th[W_] - th[L_]
        e = -coef * sigmoid(-coef * h) if kind == "DPO" else 2 * (h - 1 / (2 * coef))  # ∂mất mát/∂h
        th -= lr * (np.bincount(W_, e, 3) - np.bincount(L_, e, 3)) / 3
    return np.exp(th) / np.exp(th).sum()


for kind, coef in (("DPO", 0.1), ("DPO", 1.0), ("IPO", 0.1), ("IPO", 1.0)):
    name = "β" if kind == "DPO" else "τ"
    rows = [tabular(kind, coef, s) for s in (100, 1000, 10000)]
    print(f"{kind} {name} = {vn(coef, 1)}: " + " · ".join(f"{vn(s, 0)} bước ({', '.join(vn(v, 3) for v in q)})"
                                                       for s, q in zip((100, 1000, 10000), rows)))  # fmt: skip
c_ = 1 / (2 * 0.1)
closed = np.exp([4 * c_ / 3, 2 * c_ / 3, 0]) / np.exp([4 * c_ / 3, 2 * c_ / 3, 0]).sum()
print("IPO τ = 0,1, lời giải đóng:", ", ".join(vn(v, 3) for v in closed))
assert np.allclose(tabular("IPO", 0.1, 10000), closed, atol=1e-6)

Với DPO, cả hai β đều đi về (1, 0, 0); β chỉ đổi tốc độ. Người chấm luôn chọn câu 1, nên $-\log\sigma(\beta h)$ chỉ
về 0 khi $h \to \infty$, tức là khi $\pi(y_2), \pi(y_3) \to 0$, với mọi β. β đáng lẽ giữ chính sách gần
$\pi_{\text{ref}}$, nhưng ở đây nó chỉ xuất hiện qua tích $\beta h$, và không có điểm tối ưu hữu hạn nào để giữ. Một
mô hình phần thưởng học bằng cùng mất mát Bradley–Terry cũng đẩy chênh lệch điểm ra vô cực; Azar và cộng sự lưu ý rằng
trong thực tế nó được dừng sớm và điều chuẩn, nên phần thưởng giữ hữu hạn, và đó là thứ DPO thiếu.

Mất mát của IPO là bình phương, có cực tiểu tại $h = 1/(2\tau)$ ở từng cặp, nên chính sách dừng ở một khoảng cách hữu
hạn: (0,964; 0,034; 0,001) với τ = 0,1, đúng lời giải đóng, và (0,448; 0,321; 0,230) với τ = 1. Ở IPO, τ thật sự điều
khiển khoảng cách đến $\pi_{\text{ref}}$. Xu và cộng sự (2024) chỉ ra thêm rằng tập nghiệm của DPO chứa cả những
chính sách dồn xác suất vào câu trả lời không có trong dữ liệu.

### DPO trên thơ lục bát

Ô dưới chạy DPO β = 0,1 một lượt trên đúng 1.000 cặp mà mô hình phần thưởng ở mục 6 đã học (16 bước, lô 64), và theo
dõi log-xác suất của câu được chọn và câu bị loại, so với lúc đầu.

### 🤔 Dự đoán trước
DPO làm $h$ tăng: câu được chọn phải hơn câu bị loại. Vậy $\log\pi(y_w)$ của các câu được chọn tăng hay giảm? Trên
các cặp *huấn luyện* và trên 500 cặp *kiểm định* (chưa từng thấy) có giống nhau không?

In [ ]:
def ordered(a, b, y):
    yy = y.bool()[:, None]
    return torch.where(yy, a, b), torch.where(yy, b, a)


def dpo(win, lose, kind, beta, seed, epochs, lr=3e-4, batch=64, evals_per_epoch=2, held=None):
    torch.manual_seed(seed)
    pol = copy.deepcopy(sft)
    for p_ in pol.parameters():
        p_.requires_grad_(True)
    opt = torch.optim.Adam(pol.parameters(), lr)
    g = torch.Generator().manual_seed(60_000 + seed)
    with torch.no_grad():
        rw, rl, hw, hl = seq_logp(sft, win), seq_logp(sft, lose), seq_logp(sft, held[0]), seq_logp(sft, held[1])
    steps_per_epoch = -(-len(win) // batch)
    marks = {round(steps_per_epoch * k / evals_per_epoch) for k in range(1, evals_per_epoch * epochs + 1)}
    log, step = [], 0

    def record():
        with torch.no_grad():
            dw, dl = seq_logp(pol, held[0]) - hw, seq_logp(pol, held[1]) - hl
            log.append({"step": step, "chosen": (seq_logp(pol, win) - rw).mean().item(),
                        "rejected": (seq_logp(pol, lose) - rl).mean().item(), "held_chosen": dw.mean().item(),
                        "held_rejected": dl.mean().item(), "held_acc": (dw > dl).float().mean().item()})  # fmt: skip

    record()
    for _ in range(epochs):
        for j in torch.randperm(len(win), generator=g).split(batch):
            h = (seq_logp(pol, win[j]) - rw[j]) - (seq_logp(pol, lose[j]) - rl[j])
            loss = -F.logsigmoid(beta * h).mean() if kind == "dpo" else ((h - 1 / (2 * beta)) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
            if step in marks:
                record()
    return pol, log


win, lose = ordered(a_tr, b_tr, y_tr)
_, dlog = dpo(win, lose, "dpo", cfg["dpoBeta"], 0, cfg["dpoEpochs"], evals_per_epoch=cfg["dpoEvals"],
              held=ordered(*pairs["val"]))  # fmt: skip
print("bước   Δ log π: được chọn   bị loại   | kiểm định: được chọn   bị loại   h > 0")
for e in dlog:
    print(
        f"{e['step']:4}   {vn(e['chosen'], 3):>17}   {vn(e['rejected'], 3):>7}   | {vn(e['held_chosen'], 3):>19}"
        f"   {vn(e['held_rejected'], 3):>7}   {pct(e['held_acc'])}"
    )
assert abs(dlog[-1]["chosen"] - RUNS["live"]["dpo"][-1]["chosen"]) < 0.02, "khác với kết quả đã lưu"

Trên cặp huấn luyện, log π của câu được chọn chỉ tăng chút ít (0,085 nat sau 16 bước) còn câu bị loại giảm mạnh
(−0,469): gần hết mức tăng của $h$ đến từ việc đẩy câu bị loại xuống. Trên cặp kiểm định, *cả hai* đều giảm (−0,070 và
−0,174): câu được chọn mà mô hình chưa thấy cũng mất xác suất. DPO chỉ ràng buộc hiệu $h$, không ràng buộc từng vế.
Pal và cộng sự (2024) chỉ ra rằng DPO có thể làm giảm xác suất của câu được chọn, nhất là khi hai câu chỉ khác nhau vài
token. Razin và cộng sự (2024) gọi đây là *dịch chuyển khả năng* (likelihood displacement) và thấy nó mạnh nhất khi
biểu diễn của câu được chọn và câu bị loại giống nhau, như hai câu bát cho cùng một lời nhắc ở đây. Sau một
lượt, chính sách xếp đúng 54,8% cặp kiểm định, mới hơn đoán bừa một chút.

Script chạy DPO (β 0,03 / 0,1 / 0,3) và IPO (τ 0,01 / 0,03 / 0,1) 6 lượt trên 4.000 cặp, 3 seed, đo gold và KL trên
lời nhắc kiểm tra hai lần mỗi lượt. Hình trái đặt chúng cạnh PPO β = 0,1 và best-of-N trên cùng trục gold–√KL. Hình phải
vẽ log-xác suất của câu được chọn và câu bị loại (cặp huấn luyện) qua các lượt của DPO β = 0,1.

In [ ]:
def dpo_rows(**kw):
    return [x for x in RUNS["dpo"] if all(x[k] == v for k, v in kw.items())]


fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
specs = [({"kind": "dpo", "beta": 0.1, "n": 4000}, BLUE, "o", "DPO β = 0,1"),
         ({"kind": "dpo", "beta": 0.03, "n": 4000}, GREEN, "^", "DPO β = 0,03"),
         ({"kind": "ipo", "beta": 0.1, "n": 4000}, ORANGE, "s", "IPO τ = 0,1")]  # fmt: skip
for kw, color, marker, label in specs:
    for row in dpo_rows(**kw):
        axes[0].plot(np.sqrt([max(e["kl"], 0) for e in row["log"]]), [e["gold"] for e in row["log"]], color=color,
                     marker=marker, ms=3, lw=0.8, label=label if row["seed"] == 0 else None)  # fmt: skip
for row in rl_rows(algo="ppo", reward="rm", n=4000, beta=0.1):
    axes[0].plot(np.sqrt([max(e["kl"], 0) for e in row["log"]]), [e["gold"] for e in row["log"]], color="black",
                 marker="x", ms=3, lw=0.8, label="PPO β = 0,1" if row["seed"] == 0 else None)  # fmt: skip
bon = [b["curves"]["rm4000"] for b in RUNS["bon"]]
axes[0].plot(np.sqrt([e["klBound"] for e in bon[0]]), seeds_of(bon, "gold").mean(0), color=GREY, ls="--",
             label="best-of-N (trung bình seed)")  # fmt: skip
axes[0].set(xlabel="√KL(π‖π_ref)", ylabel="điểm gold", title="Cùng 4.000 cặp so sánh, ba cách dùng")
axes[0].legend(frameon=False, fontsize=7)
for row in dpo_rows(kind="dpo", beta=0.1, n=4000):
    ep = [e["epoch"] for e in row["log"]]
    axes[1].plot(ep, [e["chosen"] for e in row["log"]], color=BLUE, marker="o", ms=3,
                 label="câu được chọn" if row["seed"] == 0 else None)  # fmt: skip
    axes[1].plot(ep, [e["rejected"] for e in row["log"]], color=ORANGE, marker="s", ms=3, ls="--",
                 label="câu bị loại" if row["seed"] == 0 else None)  # fmt: skip
axes[1].axhline(0, color=GREY, lw=0.8)
axes[1].set(xlabel="lượt", ylabel="Δ log π so với π_ref (nat)", title="DPO β = 0,1 trên cặp huấn luyện")
axes[1].legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.show()


def span(end, key, nd):
    return f"{vn(min(e[key] for e in end), nd)}–{vn(max(e[key] for e in end), nd)}"


for kind, beta, n in sorted({(x["kind"], x["beta"], x["n"]) for x in RUNS["dpo"]}, key=lambda k: (k[0], k[2], k[1])):
    end = [r_["log"][-1] for r_ in dpo_rows(kind=kind, beta=beta, n=n)]
    best = [max(e["gold"] for e in r_["log"]) for r_ in dpo_rows(kind=kind, beta=beta, n=n)]
    print(f"{kind.upper()} {'β' if kind == 'dpo' else 'τ'} = {vn(beta, 2)}, {vn(n, 0)} cặp: gold cao nhất "
          f"{vn(min(best), 3)}–{vn(max(best), 3)}; ở lượt 6: gold {span(end, 'gold', 3)}, KL {span(end, 'kl', 1)}")  # fmt: skip
    print(f"    lượt 6: Δ log π được chọn {span(end, 'chosen', 1)}, bị loại {span(end, 'rejected', 1)}; xếp đúng cặp "
          f"kiểm định {pct(min(e['held_acc'] for e in end))}–{pct(max(e['held_acc'] for e in end))}")  # fmt: skip

Hình phải: dịch chuyển khả năng kéo dài suốt 6 lượt. Với β = 0,1, câu được chọn mất 6,2–7,7 nat, ngay trên cặp huấn
luyện, còn câu bị loại mất 16,2–20,6. β càng nhỏ, cả hai càng rơi sâu và KL càng lớn. Phần xác suất ấy đến những câu
không có trong cặp nào, mà không ai chấm. Với β = 0,03 và IPO τ = 0,01, gold ở lượt 6 tụt xuống dưới SFT: không có mô
hình phần thưởng nào để lách, nhưng DPO vẫn tối ưu quá tay. Rafailov và cộng sự (2024) đo cùng hiện tượng ở các thuật
toán căn chỉnh trực tiếp.

Hình trái: cùng 4.000 cặp, PPO β = 0,1 đạt 0,787–0,798 khi KL ≤ 4. Gold cao nhất của DPO và IPO thấp hơn: 0,742–0,749
với DPO β = 0,3 (KL 2,4–3,7 ở lượt 6), 0,739–0,747 với IPO τ = 0,1, và 0,721–0,748 với DPO β = 0,1, rồi còn 0,624–0,714
ở lượt 6. Sau 6 lượt, DPO β = 0,1 xếp đúng 54,7–63,1% cặp kiểm định, kém mô hình phần thưởng học từ cùng 4.000 cặp
(67,5–68,2% trên cặp kiểm tra). DPO học từ những cặp câu của $\pi_{\text{ref}}$; khi chính sách đã rời xa $\pi_{\text{ref}}$,
nó không có tín hiệu nào về chính những câu nó đang viết. PPO lấy mẫu câu mới ở mỗi vòng và hỏi mô hình phần thưởng,
mà mô hình phần thưởng thì khái quát từ 4.000 cặp ra mọi câu. Xu và cộng sự (2024) cũng thấy PPO được tinh chỉnh kỹ
thắng DPO trên nhiều bài; DPO lặp (lấy mẫu mới từ chính sách, gán nhãn lại) thu hẹp khoảng cách, nhưng ở đây ta chỉ chạy
DPO ngoại tuyến. Nhiều cặp hơn cũng giúp DPO: với β = 0,1, gold cao nhất là 0,699–0,717 với 1.000 cặp và 0,755–0,766
với 16.000 cặp.

## 11 · Phần thưởng kiểm chứng được (RLVR)

Nếu chất lượng *viết được* thành chương trình, như đáp án toán hay bài kiểm tra của code, ta bỏ qua mô hình phần thưởng
và tối ưu thẳng theo chương trình đó. Đó là RLVR của Tülu 3 và cách DeepSeek-R1 học suy luận. Ở đây chương trình là
chính gold. Hình dưới so sánh PPO theo gold (β = 0 và 0,01) với PPO theo mô hình phần thưởng 4.000 cặp: điểm gold, tỉ
lệ câu có chữ 6 vần với lời nhắc (phép kiểm tra khó nhất), và distinct-2, thước đo mà gold không chấm.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
specs = [({"reward": "gold", "beta": 0.0}, "black", "x", "gold, β = 0"),
         ({"reward": "gold", "beta": 0.01}, GREEN, "^", "gold, β = 0,01"),
         ({"reward": "rm", "beta": 0.01}, BLUE, "o", "mô hình phần thưởng, β = 0,01")]  # fmt: skip
for kw, color, marker, label in specs:
    rows = rl_rows(algo="ppo", n=4000, **kw)
    its = [e["it"] for e in rows[0]["log"]]
    for ax, key in zip(axes, ("gold", "rhyme", "distinct2")):
        v = np.array([[e["checks"][5] if key == "rhyme" else e[key] for e in r_["log"]] for r_ in rows])
        ax.plot(its, v.mean(0), color=color, marker=marker, ms=3, label=label)
        ax.fill_between(its, v.min(0), v.max(0), color=color, alpha=0.15)
    end = [r_["log"][-1] for r_ in rows]
    print(
        f"{label:29}: vòng 200 gold {vn(min(e['gold'] for e in end), 3)}–{vn(max(e['gold'] for e in end), 3)}, "
        f"KL {vn(min(e['kl'] for e in end), 1)}–{vn(max(e['kl'] for e in end), 1)}, distinct-2 "
        f"{vn(min(e['distinct2'] for e in end), 2)}–{vn(max(e['distinct2'] for e in end), 2)}, vần "
        f"{pct(min(e['checks'][5] for e in end))}–{pct(max(e['checks'][5] for e in end))}"
    )
for ax, t_ in zip(axes, ("Điểm gold", "Vần chữ 6", "Distinct-2")):
    ax.set(xlabel="vòng", title=t_)
axes[0].legend(frameon=False, fontsize=7)
vn_axes(fig)
plt.show()
row = rl_rows(algo="ppo", reward="gold", n=4000, beta=0.0, seed=0)[0]
print("\n".join(row["lines"]["200"][:4]))
coda_i = (CODA[X_test[:, 5]] == codas.index("i")).float().mean().item()
print(f"lời nhắc kiểm tra có chữ 6 tận cùng bằng âm cuối i (như “lời”, “nơi”): {pct(coda_i)}")

Theo chính gold thì không có gì để lách: gold lên 0,888–0,893 với β = 0, cao hơn mọi mô hình phần thưởng. Nhưng hãy
đọc các câu: "có nơi lời có lời lời lại nơi". Câu đúng luật mà vô nghĩa, chỉ lặp vài từ (distinct-2 0,16–0,29). Gold
chấm luật chứ không chấm nghĩa, nên RL tối ưu đúng thứ được chấm, không hơn. Với β = 0,01, câu đa dạng hơn hẳn
(distinct-2 0,69–0,91) và gold đạt 0,795–0,890.

Vần chữ 6 dừng ở đúng 26,8% trên cả ba seed, đúng bằng tỉ lệ lời nhắc kiểm tra có chữ 6 tận cùng bằng âm cuối i
(26,8%). Chính sách học một mẹo: đặt "lời" ở chữ 6 và "nơi" ở chữ 8. Theo luật vần thông, mọi âm tiết tận cùng bằng i
vần với nhau, nên mẹo này thắng ở mọi lời nhắc tận cùng bằng i, và "lời" (huyền) với "nơi" (ngang) cũng qua luật chữ
6–8. Với những lời nhắc khác, chính sách không tìm ra cách vần, vì $\pi_{\text{ref}}$ hiếm khi vần (12,8%) và mỗi vần
cần những từ khác nhau. RL khuếch đại những gì $\pi_{\text{ref}}$ đôi khi đã làm; nó khó tìm ra điều $\pi_{\text{ref}}$
gần như không bao giờ làm. Yue và cộng sự (2025) đặt cùng câu hỏi cho RLVR trên toán: với k lớn, pass@k của mô hình gốc
bằng hoặc hơn mô hình đã qua RL; câu hỏi đó vẫn đang được tranh luận. Mô hình phần thưởng (β = 0,01) chỉ đạt vần
11,9–14,8%, gần mức SFT: nó không thấy luật vần nên không thưởng.

Trong thực tế, phần thưởng kiểm chứng được dùng ở nơi đúng sai rõ ràng: Tülu 3 chạy RLVR sau DPO, chỉ trên những bài
có đáp án kiểm tra được; DeepSeek-R1 dùng phần thưởng theo luật cho toán và code, và mô hình phần thưởng cho phần còn
lại.

## 12 · Thử thay đổi

### Đuôi của sai số

Mục 7 so sánh sai số chuẩn với Student-t ν = 1,5. Ô dưới thử những đuôi ở giữa: ν càng lớn, Student-t càng gần phân
phối chuẩn. Bạn có thể sửa danh sách rồi chạy lại.

In [ ]:
for nu in (1.5, 3, 10):
    r_try = g_tab + 0.6 * np.random.default_rng(1).standard_t(nu, K)
    gb_ = np.array([best_of(r_try, N) @ g_tab for N in NS_TAB])
    print(
        f"ν = {vn(nu, 1):>4}: best-of-N gold cao nhất {vn(gb_.max(), 2)} ở N = {vn(NS_TAB[gb_.argmax()], 0)}, "
        f"ở N = 4.096: {vn(gb_[-1], 2)}"
    )

### Người chấm tinh mắt hơn hay ồn hơn

Đổi $s$ đổi cả trần độ chính xác lẫn mức đồng ý giữa hai người chấm. Ô dưới tính cả hai trên 1.000 cặp huấn luyện.

In [ ]:
for s_ in (3, 10, 30, 100):
    p_ = torch.sigmoid(s_ * (ga - gb))
    print(
        f"s = {s_:3}: hai người chấm đồng ý {pct((p_**2 + (1 - p_) ** 2).mean())}, "
        f"trần {pct(torch.maximum(p_, 1 - p_).mean())}"
    )

### KL trong bài toán bandit với β khác

Ô dưới tính hai đích của mục 8, $\pi^*$ (KL trong phần thưởng) và $\pi_{\text{rev}}$ ($k_3$ trong mất mát), bằng lời
giải đóng cho vài β khác, không cần huấn luyện. Muốn xem chính sách học được, đổi `beta_b` ở mục 8 rồi chạy lại các ô
từ đó.

In [ ]:
for beta_try in (0.03, 0.1, 0.3):
    s_star, s_rev = torch.softmax(lp_ref_b + r_band / beta_try, 0), pi_rev(beta_try)
    print(
        f"β = {vn(beta_try, 2)}: TV(π*, π_rev) {vn(tv(s_star, s_rev), 3)} · xác suất lớn nhất: π* {pct(s_star.max())}, "
        f"π_rev {pct(s_rev.max())}"
    )
    if beta_try == beta_b:
        assert abs(tv(s_star, s_rev) - tv(star, rev)) < 1e-9

## 13 · Giới hạn

- **Gold là một chương trình 7 luật, không phải gu của người.** Nhờ vậy ta đo được Goodhart chính xác, nhưng chỗ mù của
  mô hình phần thưởng ở đây đơn giản (hai luật bị bỏ qua). Mô hình phần thưởng thật sai theo những cách đa dạng và khó
  thấy hơn.
- **Người chấm mô phỏng có một tham số** $s$, cộng một biến thể thích câu dài. Người thật mệt, không đồng nhất, có quan
  điểm riêng, và đôi khi bị câu trả lời trôi chảy đánh lừa.
- **Mô hình rất nhỏ:** 254.016 tham số và 1.300 cặp câu, nên câu không có nghĩa và ta chỉ đo được luật. Ở mô hình lớn,
  mô hình phần thưởng khởi đầu từ một mô hình đã hiểu ngôn ngữ và khái quát tốt hơn nhiều.
- **Một bộ siêu tham số PPO**, chọn trước theo tỉ lệ bị kẹp, không dò. Kết luận "phạt KL nâng đường gold–KL" có thể
  đổi với bộ khác.
- **DPO chỉ chạy ngoại tuyến.** DPO lặp và các biến thể khác (KTO, SimPO, ORPO) không có ở đây; GRPO có trong Bài 26.
- **3 seed.** Nhiều khoảng chồng nhau, và những kết luận đó được ghi là trong dao động giữa các seed.

## Tóm tắt
- **RLHF** gồm ba bước: SFT, rồi một mô hình phần thưởng Bradley–Terry học từ các cặp so sánh, rồi RL (PPO) với phạt
  KL đến $\pi_{\text{ref}}$. Bradley–Terry chỉ xác định phần thưởng đến một hằng số cho mỗi lời nhắc.
- **Nhãn so sánh vốn ồn:** trên 2.000 cặp kiểm tra của script, trần độ chính xác là 78,0%, và hai người chấm chỉ đồng
  ý 73,2%. Mô hình phần thưởng 4.000 cặp đoán đúng 67,5–68,2% nhãn của các cặp ấy, nhưng gần như bỏ qua luật vần và
  luật huyền/ngang.
- **PPO không phạt tìm ra chỗ mù đó:** "một tra … tra", proxy 1,79, gold chỉ 0,740, KL 43,4 nat. Gold đạt đỉnh rồi
  giảm trong khi proxy vẫn tăng.
- **Sai số đuôi nặng** làm gold của best-of-N và $\pi_\beta$ tụt dưới cả $\pi_{\text{ref}}$ (Goodhart thảm họa). Trong
  bài này, phạt KL β = 0,1 cho gold cao hơn β = 0 ở cùng ngân sách KL.
- **Đặt KL ở đâu là quan trọng:** trong phần thưởng thì chính sách về đúng $\pi^*$; $k_1$ trong mất mát không phạt gì
  cả; $k_3$ trong mất mát phạt KL theo chiều ngược lại.
- **DPO** bỏ mô hình phần thưởng, nhưng không có điểm tối ưu hữu hạn khi sở thích tuyệt đối (IPO thì có). Trên thơ, nó
  hạ cả xác suất của câu được chọn (dịch chuyển khả năng), và chạy ngoại tuyến thì thua PPO ở cùng KL.
- **Phần thưởng kiểm chứng được** không lách được, nhưng RL tối ưu đúng thứ được chấm: câu đúng luật, vô nghĩa, và chỉ
  vần khi đã có một mẹo mà $\pi_{\text{ref}}$ đôi khi dùng.

## Câu hỏi ôn tập
1. Mô hình phần thưởng 4.000 cặp chỉ đoán đúng 67,5–68,2% nhãn. Vì sao con số đó không tệ như nghe?
2. Cộng 5 vào điểm của mọi câu trả lời cho một lời nhắc. Mất mát Bradley–Terry đổi thế nào? PPO và RLOO có bị ảnh
   hưởng không?
3. Trong hình best-of-N đuôi nặng ở mục 7, vì sao gold giảm khi N tăng, dù KL của best-of-N không quá
   $\log N - (N-1)/N$?
4. Một thuật toán cộng $\beta\,k_3$ vào hàm mất mát. Nó hướng chính sách về phân phối nào? Khác $\pi^*$ của RLHF ra
   sao?
5. Vì sao DPO với sở thích tuyệt đối đưa chính sách về (1, 0, 0) với mọi β, còn IPO thì không?
6. Chính sách PPO β = 0 viết "một tra … tra". Hãy giải thích vì sao câu này đạt năm luật mà không đạt luật huyền/ngang,
   rồi đề xuất hai cách ngăn.

<details><summary>Gợi ý đáp án</summary>

1. Nhãn ồn: hai câu hơn nhau một luật chỉ được chọn đúng với xác suất 0,807, và câu hòa điểm thì nhãn là tung đồng
   xu. Trần của mọi mô hình là 78,3% trên 1.000 cặp kiểm tra của notebook (mô hình biết hết luật đạt 79,4% trên mẫu
   này) và 78,0% trên 2.000 cặp kiểm tra của script, và hai người chấm chỉ đồng ý 73,2%. Con số 67,5–68,2% được đo trên
   2.000 cặp ấy: hãy đọc nó so với trần 78,0%, không so với 100%.
2. Mất mát không đổi, vì nó chỉ dùng hiệu điểm của hai câu cho cùng lời nhắc. RLOO trừ trung bình của các câu khác cho
   cùng lời nhắc, nên hằng số bị trừ hẳn. PPO trừ giá trị $V(s_t)$; khi mô hình giá trị học được hằng số đó, lợi thế
   không đổi. Vì vậy notebook chuẩn hóa điểm chung, không cần chuẩn hóa theo lời nhắc.
3. Với đuôi nặng, câu thắng trong N câu là câu có sai số lớn nhất chứ không phải câu có gold cao nhất, và N càng lớn
   thì càng chắc chọn được một sai số khổng lồ. KL nhỏ chỉ nói chính sách gần $\pi_{\text{ref}}$; một lượng xác suất
   nhỏ đặt đúng vào những câu ấy đủ đẩy proxy lên cao (Kwa và cộng sự, 2024).
4. Kỳ vọng của gradient là $\nabla\,\mathrm{KL}(\pi_{\text{ref}}\,\|\,\pi)$, nên chính sách hướng về
   $\pi_{\text{rev}} = \beta\,\pi_{\text{ref}}/(\lambda - r)$, không phải $\pi^* \propto \pi_{\text{ref}}\,e^{r/\beta}$:
   phần thưởng vào qua một phân thức thay cho hàm mũ. Trong bài toán bandit, khoảng cách TV giữa hai phân phối là 0,520.
5. Với DPO, mất mát của mỗi cặp giảm mãi khi $h$ tăng, nên điểm tối ưu nằm ở vô cực với mọi β. IPO phạt bình phương
   độ lệch của $h$ so với $1/(2\tau)$, nên mỗi cặp có một $h$ tối ưu hữu hạn.
6. "Tra" thanh ngang, đứng ở chữ 2, 6, 8 nên đạt các luật về độ dài và thanh mà mô hình phần thưởng thưởng. Chữ 6 và
   chữ 8 cùng thanh ngang, nên luật huyền/ngang luôn trượt, nhưng mô hình phần thưởng gần như không thấy luật đó (hệ số
   −0,02). Cách ngăn: phạt KL (β = 0,1 giữ KL 1,9–3,7); thu thêm nhãn trên chính câu của chính sách rồi học lại mô hình
   phần thưởng, như InstructGPT và Llama 3 làm qua nhiều vòng; thêm một bộ kiểm tra theo luật cho phần kiểm chứng được.
</details>

## Tìm hiểu thêm
- Christiano và cộng sự (2017), *Deep reinforcement learning from human preferences*:
  [arXiv:1706.03741](https://arxiv.org/abs/1706.03741).
- Ziegler và cộng sự (2019), *Fine-Tuning Language Models from Human Preferences*:
  [arXiv:1909.08593](https://arxiv.org/abs/1909.08593).
- Stiennon và cộng sự (2020), *Learning to summarize from human feedback*:
  [arXiv:2009.01325](https://arxiv.org/abs/2009.01325).
- Ouyang và cộng sự (2022), *Training language models to follow instructions with human feedback* (InstructGPT):
  [arXiv:2203.02155](https://arxiv.org/abs/2203.02155).
- Bai và cộng sự (2022), *Training a Helpful and Harmless Assistant with RLHF*:
  [arXiv:2204.05862](https://arxiv.org/abs/2204.05862); *Constitutional AI*:
  [arXiv:2212.08073](https://arxiv.org/abs/2212.08073).
- Schulman và cộng sự (2017), *Proximal Policy Optimization Algorithms*:
  [arXiv:1707.06347](https://arxiv.org/abs/1707.06347); GAE (2016): [arXiv:1506.02438](https://arxiv.org/abs/1506.02438).
- Huang và cộng sự (2024), *The N+ Implementation Details of RLHF with PPO*:
  [arXiv:2403.17031](https://arxiv.org/abs/2403.17031).
- Gao, Schulman & Hilton (2022), *Scaling Laws for Reward Model Overoptimization*:
  [arXiv:2210.10760](https://arxiv.org/abs/2210.10760).
- Kwa, Thomas & Garriga-Alonso (2024), *Catastrophic Goodhart: regularizing RLHF with KL divergence does not mitigate
  heavy-tailed reward misspecification*: [arXiv:2407.14503](https://arxiv.org/abs/2407.14503).
- Coste và cộng sự (2024), *Reward Model Ensembles Help Mitigate Overoptimization*:
  [arXiv:2310.02743](https://arxiv.org/abs/2310.02743).
- Singhal và cộng sự (2023), *A Long Way to Go: Investigating Length Correlations in RLHF*:
  [arXiv:2310.03716](https://arxiv.org/abs/2310.03716).
- Rafailov và cộng sự (2023), *Direct Preference Optimization*: [arXiv:2305.18290](https://arxiv.org/abs/2305.18290).
- Azar và cộng sự (2023), *A General Theoretical Paradigm to Understand Learning from Human Preferences* (IPO):
  [arXiv:2310.12036](https://arxiv.org/abs/2310.12036).
- Pal và cộng sự (2024), *Smaug: Fixing Failure Modes of Preference Optimisation with DPO-Positive*:
  [arXiv:2402.13228](https://arxiv.org/abs/2402.13228).
- Razin và cộng sự (2024), *Unintentional Unalignment: Likelihood Displacement in DPO*:
  [arXiv:2410.08847](https://arxiv.org/abs/2410.08847).
- Rafailov và cộng sự (2024), *Scaling Laws for Reward Model Overoptimization in Direct Alignment Algorithms*:
  [arXiv:2406.02900](https://arxiv.org/abs/2406.02900).
- Xu và cộng sự (2024), *Is DPO Superior to PPO for LLM Alignment?*: [arXiv:2404.10719](https://arxiv.org/abs/2404.10719).
- Ahmadian và cộng sự (2024), *Back to Basics: Revisiting REINFORCE Style Optimization for Learning from Human
  Feedback in LLMs* (RLOO): [arXiv:2402.14740](https://arxiv.org/abs/2402.14740).
- Tang & Munos (2025), *On a few pitfalls in KL divergence gradient estimation for RL*:
  [arXiv:2506.09477](https://arxiv.org/abs/2506.09477).
- Grattafiori và cộng sự (2024), *The Llama 3 Herd of Models*: [arXiv:2407.21783](https://arxiv.org/abs/2407.21783).
- Lambert và cộng sự (2024), *Tülu 3*: [arXiv:2411.15124](https://arxiv.org/abs/2411.15124).
- Shao và cộng sự (2024), *DeepSeekMath* (GRPO): [arXiv:2402.03300](https://arxiv.org/abs/2402.03300).
- DeepSeek-AI (2025), *DeepSeek-R1*: [arXiv:2501.12948](https://arxiv.org/abs/2501.12948).
- Yue và cộng sự (2025), *Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base
  Model?*: [arXiv:2504.13837](https://arxiv.org/abs/2504.13837).
- MacDiarmid và cộng sự (2025), *Natural Emergent Misalignment from Reward Hacking in Production RL*:
  [arXiv:2511.18397](https://arxiv.org/abs/2511.18397).
- Nathan Lambert, *Reinforcement Learning from Human Feedback* (sách mở): [rlhfbook.com](https://rlhfbook.com/).

**Quay lại:** [Bài 26 · Từ mô hình ngôn ngữ đến trợ lý](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)